# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** どこから読み、最初に何をするか |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# 診断の報告(2026-09-14)

**添付の脆弱性カタログ(`security-review-vulnerability-catalog.md`、2026-09-13 付)を物差しにして、
Website(server/)と Android アプリ(Test/)を調べ、見つけたものと直したものをまとめた。**
あわせて、同じ日に頼まれた改修 —— セキュリティの向上・ドメインを .env で変える・Let's Encrypt の更新・負荷の対処・地図の正本を2つに分ける —— の結果も「直したこと」として載せる。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**
> **直したものはどれもまだ本番に入っていない(未配備・未配布)。** 本番で実行する順番は §6。

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 0. この報告の読み方

### 0.1 目次

| 節 | 中身 |
|---|---|
| §1 | **要約。** 件数・いちばん大事な5つ・本番でまだ済んでいないこと |
| §2 | **カタログの節ごとの結果。** ID ごとに「問題なし」「該当なし」「所見あり」「未確認」 |
| §3 | **所見の一覧。** Web(W-xx)と Android(A-xx)。直したかどうか |
| §4 | **本番ホストの実測。** 読むだけで確かめた値 |
| §5 | **直したこと。** セキュリティ・ドメイン・Let's Encrypt・負荷・地図の配信。手元で確かめたこと |
| §6 | **本番で実行すること。** 順番どおりのセル(1〜25) |
| §7 | **利用者の判断待ち** |
| §8 | **この診断の限界** |

### 0.2 確かさの区別

所見には2つの軸がある。**「本当にあるか」(判定)と「どれくらい痛いか」(深刻度)。**

| 判定 | 意味 |
|---|---|
| **確認済み** | 見つけた担当とは別の検証の担当が、コード(と必要なら SDK のソース)を読んで成り立つと確かめた |
| **可能性** | 筋は通るが、決め手(本番の設定・実機の挙動)を確かめていない。何が決め手かは §3 の備考 |
| **反証** | 検証で成り立たないと分かった。一覧には残すが、件数の「直す対象」には入れない |
| **未検証** | 最後の見直し役が足した指摘で、検証の担当を通していない。**確度はほかより低い**(本番の実測で裏が取れたものは備考に書いた) |
| **本番で確認** | 本番ホストを**読むだけで**実測して見つかったもの(H 系。§4) |

深刻度は **カタログの区分(重大・高・中・低・情報)を、この環境の到達性に合わせて付け直した値**。
見つけた時点の値から検証で下げたものが多い(例: web の侵害が前提になるものは「中」→「低」)。
**「重大」「高」は1件も残らなかった。** ただし、それは「この構成とこの読み方で」という条件付き(§8)。

「直したか」の書き方:

| 書き方 | 意味 |
|---|---|
| **直した(未配備)** | 手元(server/)は直した。**本番にはまだ入っていない** |
| **直した(未配布)** | Android のソースは直し、テストも通した。**利用者へはまだ配っていない** |
| **一部** | 約束の範囲だけ直した。残りは備考 |
| **未対応** | 手を付けていない(理由を書いた) |
| **利用者の判断待ち** | 直し方に選択肢があるか、手元のファイルを動かす話。§7 |
| **対応不要** | 本番の実測で当てはまらないと分かった |

**file:line は診断した時点(直す前)の行番号。** 直した後のファイルでは行がずれている。

### 0.3 カタログそのものの限界(正直に書いておく)

カタログは物差しとして役に立ったが、**そのまま事実として扱えない箇所がある**。

1. **2026 年の CVE 番号・CERT/CC VU#492466・JVNVU#99418634・修正版の番号(Logto 1.41.0、logto-tunnel 0.3.9)は、この診断では一次情報で確かめていない。**
   この PC から NVD・GitHub Security Advisory・CERT/CC には接続していない。カタログ自身も CVSS の値の一部を集約サイト(二次情報)経由と書いている。
   **Logto の版を上げる判断や、LG 系を「該当なし」とした判断を人に説明するときは、先に一次情報を読む。**
2. カタログの §2 の件数(合計 90 件)は、表に並ぶ ID の数(WA 70・DB 22・LG 10・LF 18・PM 17・AD 9・DK 13・NG 10・MP 4・SK 6・OS 8 = **187**)と合わない。件数の表は当てにしない。
3. 表の中の相互参照に食い違いがある。WA-40(XML 注入)の「実例 LG-03」は、内容からは LG-07(SAML への XML 注入)。WA-59(戻り値の未チェック)の「実例 LG-04」は、内容からは LG-03(削除エラーの黙殺)。
4. カタログは本人も書いているとおり**チェックリストであって、この環境の診断結果ではない**。重大度は「典型的な構成での目安」で、この報告では到達性を見て付け直した。

## 1. 要約

### 1.1 件数

**診断の所見 62 件 + 本番の実測で見つけたもの 9 件 = 71 件。** 重大・高は 0 件。
Web と Android は**所見の場所で分けた**(Android の担当が見つけたものでも、場所がサーバーなら Web に入れた —— W-05 と W-35)。

**Web(43 件)**

| 深刻度 | 確認済み | 可能性 | 未検証 | 本番で確認 | 計 |
|---|---|---|---|---|---|
| 中 | 2 | 1 | 0 | 2 | **5** |
| 低 | 15 | 3 | 5 | 7 | **30** |
| 情報 | 7 | 0 | 1 | 0 | **8** |
| 計 | 24 | 4 | 6 | 9 | **43** |

**Android(28 件)**

| 深刻度 | 確認済み | 可能性 | 反証 | 計 |
|---|---|---|---|---|
| 中 | 1 | 1 | 0 | **2** |
| 低 | 17 | 0 | 0 | **17** |
| 情報 | 8 | 0 | 1 | **9** |
| 計 | 26 | 1 | 1 | **28** |

**直したかどうか**

| | 直した | 一部 | 未対応 | 利用者の判断待ち | 対応不要 | 反証 |
|---|---|---|---|---|---|---|
| Web | 26 | 10 | 4 | 2 | 1 | 0 |
| Android | 23 | 0 | 1 | 3 | 0 | 1 |

同じ件を別の担当が重ねて見つけたもの: A-03 と A-09(設定の取り込みの型)、A-23 と A-26(build (1).gradle.kts)、W-31 は W-16 と W-09 に近い。件数はそのまま数えた。

### 1.2 いちばん大事な5つ

| # | 何か | 番号 | いま |
|---|---|---|---|
| 1 | **アプリが地図を取れない。** 本番の配信設定にあるアクセスコードは、配信先の無い TEST1 の1件だけ | W-02 | 直した(未配備)。**配備の後、管理画面で付け替える(§6 の 21)** |
| 2 | **Logto Console(admin テナント)に MFA もパスワード方針も総当たりロックも無い。** IP 制限とゲート(default テナントの MFA)の内側だが、認証基盤の全権 | W-01 | 見張りを足した。**切り替えは §6 の 23、進め方は §7** |
| 3 | **PATH_INFO 付きの URL で nginx の守りが外れる。** 本番で `/logto-client.php/x` が 200、管理画面の HTML に 30 日の public キャッシュ | W-13 | 直した(未配備) |
| 4 | **ランキング参加の既定が ON。** ログインした来場者の表示名が、同意なく誰でも見られるランキングに載る | A-01 | 直した(未配布)。**溜まった行をどうするかは §7** |
| 5 | **web が破られたときの広がりが大きい。** web に Postgres の資格情報、その Postgres は SUPERUSER、全コンテナが同じネットワーク、Soketi は EOL の Node 16 を root で | W-16 W-31 W-04 | 資格情報を外し、Soketi を node 利用者に。**SUPERUSER・Node 16・ネットワークの分割は §7** |

次点: 会場の共有回線でアクセスコードの失敗ロックが全員を止める W-05(直した)、fail2ban が止まっている W-06(§6 の 22)。

### 1.3 本番でまだ済んでいないこと

1. **配備そのもの。** この報告の「直した」はすべて手元だけ(§6 の 4〜9)
2. Mailpit を止めて消す(§6 の 6)、cron を版 6 にする(§6 の 8)、証明書の更新方式を webroot に揃える(§6 の 9)
3. TEST1 の付け替え(§6 の 21)
4. fail2ban を動かす(§6 の 22)、Logto Console の MFA(§6 の 23)
5. Android の両フレーバーをリリースビルドして実機で確かめ、**サーバーの配備の後に**配る(§6 の 25)
6. §7 の判断(MariaDB の root@% と Main@%、Postgres の非 SUPERUSER、Soketi の Node、Test.zip、ランキングの溜まった行 など)

## 2. カタログの節ごとの結果

**ID ごとに1行。** 根拠は、診断の担当が「見て問題なし」と書いた範囲(audit-coverage)と、本番を読むだけで測った値(§4)。
「所見あり」の番号は §3 の W-xx / A-xx。**「問題なし」は「その読み方で見つからなかった」であって、無いことの証明ではない**(§8)。

### 2.1 A01 アクセス制御の不備

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-01 垂直権限昇格 | 所見あり W-36 | admin の全ページと admin/api の全本が先頭で guard.php を読む。判定できないときは 503(fail closed)。Android の管理者判定もサーバーの応答で決まる。見つかったのは「全権がスコープ1つで決まる」設計の注意だけ |
| WA-02 IDOR | 問題なし | app-settings・app-avatar・app-ranking・account はトークンの sub だけを使う。イベントの地点は WHERE id = ? AND event_id = ? |
| WA-03 機能レベル認可 | 所見あり W-14 | account.php だけ停止判定が無かった。ほかの口は guard・logto_guard が停止も見る |
| WA-04 強制ブラウジング | 所見あり W-13 | lib・config・scripts・uploads・vendor・/admin/_ は nginx で 404。ただし PATH_INFO 付きで外れた(本番で実測) |
| WA-05 SSRF | 問題なし | curl の宛先は設定の JWKS・Logto の endpoint(リダイレクト追従なし)・reCAPTCHA の定数だけ |
| WA-06 CORS | 問題なし | Access-Control-Allow-* を出す PHP は無い。本番でも CORS ヘッダーを返していない(実測) |
| WA-07 JWT 検証 | 問題なし | JWK::parseKeySet で署名を検証し、iss・aud・client_id を照合 |
| WA-08 ディレクトリ横断 | 問題なし | floor-image は realpath と基点の前方一致、地図ファイルは / \ .. を拒否。nginx に alias は無い。Android の保存先は固定名 |
| WA-09 メタデータ操作 | 問題なし | 権限の情報を Cookie や hidden に持たない |

### 2.2 A02 セキュリティ設定の不備

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-10 既定の認証情報 | 問題なし | .env.example の秘密は空。Soketi の鍵は既定値でない・MariaDB の空パスワードは 0 件(本番で実測) |
| WA-11 管理画面の公開 | 所見あり W-03 W-01 | 8281・3002・8025 は IP 制限と Logto のゲート(MFA)の2枚重ね。IPv6 の抜け道は、本番にグローバル IPv6 が無いので今は無い |
| WA-12 詳細エラーの露出 | 所見あり W-20 | display_errors は Off だが、捕まえた例外の文面を画面に出していた |
| WA-13 ディレクトリ一覧 | 問題なし | autoindex の指定なし |
| WA-14 XXE | 問題なし | アプリのコードに XML パーサ・ZIP 展開は無い。SAML も使っていない |
| WA-15 セキュリティヘッダ | 所見あり W-07 W-10 | CSP(nonce)・HSTS・nosniff・Referrer-Policy・frame-ancestors はある。Permissions-Policy だけ無かったので足した |
| WA-16 不要なサービス | 所見あり W-25 | 本番でも Mailpit が動いていた(0 通、実測) |
| WA-17 過剰権限 | 所見あり W-19 W-24 | privileged・docker.sock のマウントは無い(実測) |
| WA-18 版の露出 | 問題なし | server_tokens off、expose_php Off、X-Powered-By を除去 |

### 2.3 A03 ソフトウェアサプライチェーン

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-19 既知脆弱性のある依存 | 未確認 | composer.lock と Gradle の版は一覧にしたが、CVE データベースと照らしていない(外部に接続していない)。Logto Android SDK はベータ版 A-25 |
| WA-20 イメージの非固定 | 所見あり W-18 | nginx・mailpit・certbot・postfix は digest、php・postgres・mariadb・phpmyadmin・logto はタグ |
| WA-21 依存関係の混乱 | 問題なし | Composer は packagist の4つだけ。Gradle は google() と mavenCentral() に限り FAIL_ON_PROJECT_REPOS |
| WA-22 タイポスクワッティング | 問題なし | 同上(名前は目で見た) |
| WA-23 ビルドパイプライン | 該当なし | CI は無い。手元の PC でビルドし配備する。その PC の鍵にパスフレーズが無い点は W-24 |
| WA-24 SBOM | 未対応 | 作っていない |
| WA-25 EOL 製品 | 所見あり W-04 | Soketi の Node 16。postgres 17・mariadb 11.4・php 8.4 は保守期間内 |

### 2.4 A04 暗号の失敗

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-26 平文通信 | 問題なし(一部未確認) | TLS 1.2/1.3 のみ(実測)、MariaDB は TLS、Android は平文禁止。web → mailserver:587 の暗号化は見ていない |
| WA-27 パスワードの保存 | 問題なし | Web 地図の合言葉とアクセスコードは password_hash。利用者のパスワードは Logto |
| WA-28 ハードコードの秘密 | 所見あり A-06 | リポジトリ・Dockerfile に秘密は無い。Test.zip の中に DB 接続設定が残る |
| WA-29 弱い TLS | 問題なし | 本番で TLS 1.0/1.1 を拒否(実測) |
| WA-30 証明書検証の無効化 | 問題なし | PHP の curl・PDO・phpMyAdmin は検証あり。Android は system CA のみ |
| WA-31 予測可能な乱数 | 問題なし | random_bytes・random_int・UUID.randomUUID |
| WA-32 保存データの非暗号化 | 所見あり W-23 A-08 | ホストの控えは CMS で暗号化 |
| WA-33 パディングオラクル | 所見あり W-42 | 控えの暗号に認証が無い(改ざんを検出できない)。オンラインで復号の成否を返す口は無い |

### 2.5 A05 インジェクション

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-34 格納型 XSS | 所見あり W-15 W-39 | テンプレートの出力は km_e で逃がしている。書き込めるのは管理者だけで、CSP の nonce でスクリプトは動かない |
| WA-35 反射型 XSS | 問題なし | 全 PHP の出力を洗った |
| WA-36 DOM ベース XSS | 所見あり W-15 | 経路表示のツールチップだけ逃がし漏れ |
| WA-37 OS コマンド | 問題なし | src に exec 系は無い。運用スクリプトに eval・Invoke-Expression は無い |
| WA-38 コード注入 | 問題なし | 同上 |
| WA-39 LDAP | 該当なし | LDAP を使っていない(個別の grep はしていない) |
| WA-40 XPath・XML | 該当なし | XML を扱う関数が 0 件 |
| WA-41 ヘッダ分割 | 問題なし | 戻り先はサイト内の絶対パスだけ。header() は改行を拒否 |
| WA-42 メールヘッダ | 問題なし | 宛先は環境変数、件名は許可リスト、利用者の値は本文だけ |
| WA-43 ログ注入 | 問題なし | 入力をそのまま error_log に渡していない |

### 2.6 A06 安全でない設計

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-44 レート制限 | 所見あり W-05 W-35 W-30 W-27 | nginx の制限が GET を一律に除外し、アプリの失敗ロックは IP 単位で原子的でなかった |
| WA-45 業務ロジック | 未確認 | 金銭は無い。ランキングの回数を水増しできるかは見ていない |
| WA-46 パスワードリセット | 該当なし | Logto に任せている。verify.php はコードを保存も検証もしない |
| WA-47 信頼境界 | 所見あり A-01 A-24 A-20 A-03 | サーバーの判定は正しいが、端末の既定・要求するスコープ・版の比較が甘かった |
| WA-48 MFA の設計上の抜け道 | 所見あり W-01 | SSO の接続は 0 件なので LG-06 の経路は無い。Console(admin テナント)に MFA が無い |

### 2.7 A07 認証の失敗

| ID | 結果 | 根拠・備考 |
|---|---|---|
| — | §2.12・§2.13 | カタログのとおり Logto とログインフォームの節で扱う |

### 2.8 A08 完全性の失敗

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-49 安全でない逆シリアル化 | 所見あり A-03 | Gson は具体クラスへの変換だけで RCE の形ではない。型の検査漏れで起動できなくなる不具合があった |
| WA-50 署名なし更新 | 未確認 | APK の同一性は Android の署名に任せる前提。配布ページにハッシュを出しているかは見ていない |
| WA-51 SRI | 問題なし | 外部スクリプトは reCAPTCHA だけ(SRI は付けられない。CSP でパスまで絞っている) |
| WA-52 署名なし Webhook | 所見あり W-17 | 署名は検証している。鍵が web に渡っておらず、常に拒否していた(安全側に閉じる不具合) |

### 2.9 A09 ログとアラートの失敗

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-53 認証イベントの記録 | 所見あり W-22 W-21 | ログイン成功だけ記録し、拒否・停止・サインアウトは残っていなかった |
| WA-54 ログへの機微情報 | 所見あり W-28 W-40 W-41 | パスワードの本文はログに出さない。/verify.php のクエリは残さない |
| WA-55 改ざん防止 | 所見あり W-22 | 監査ログは同じホストの DB だけ。外部転送は未対応 |
| WA-56 アラート | 所見あり W-06 | 控え・更新・セキュリティの定期メールはある。fail2ban の停止を知らせていなかった |
| WA-57 時刻同期 | 問題なし | timesyncd が active で同期済み(実測) |

### 2.10 A10 例外条件の処理

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-58 fail-open な認証 | 問題なし | guard は 503、gate は 401 で閉じる。Android は API 障害で来場者に倒れる |
| WA-59 戻り値の未チェック | 所見あり W-17 A-07 W-02 A-10 A-12 A-13 A-14 A-16 A-17 | — |
| WA-60 競合・TOCTOU | 所見あり W-30 A-11 A-19 | 書き込みの多くはトランザクションと tmp → rename |
| WA-61 未捕捉例外 | 所見あり A-02 A-09 | — |
| WA-62 資源の枯渇 | 所見あり W-27 A-21 A-04 A-15 A-18 A-27 W-12 | nginx の本文上限は 3m・12m・210m。正規表現に ReDoS の形は無い |

### 2.11 その他

| ID | 結果 | 根拠・備考 |
|---|---|---|
| WA-63 CSRF | 所見あり W-37 | Cookie 認証の POST はすべて副作用の前に km_csrf_verify。サインアウトだけ GET だった |
| WA-64 クリックジャッキング | 問題なし | X-Frame-Options と frame-ancestors。3002 は Logto 自身が送る(実測)ので W-29 は対応不要 |
| WA-65 オープンリダイレクト | 問題なし | 戻り先は ^/(?!/) に限定し、APP_URL を前に付ける |
| WA-66 ファイルアップロード | 問題なし | 拡張子の許可リスト・ランダム名・getimagesize・SVG 不可・nosniff・attachment |
| WA-67 キャッシュのポイズニング・デセプション | 所見あり W-43 | 共有キャッシュは無いので今は漏れない |
| WA-68 リクエストスマグリング | 問題なし | nginx が要求を組み直す標準の構成 |
| WA-69 サブドメインの乗っ取り | 未確認 | DNS のレコードを見ていない。証明書の SAN は ito8795.com だけ(実測) |
| WA-70 WebSocket の認可(CSWSH) | 所見あり W-38 | presence チャンネルは署名が要り、中身は読めない。Origin の検査が無かった(任意の Origin で 101、実測) |

### 2.12 DB 層(MariaDB・PostgreSQL)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| DB-01〜DB-07 SQL インジェクション | 問題なし | プレースホルダ。テーブル名と列名は許可リスト(kmt_ 接頭辞・実在列)、LIMIT は PARAM_INT、IN 句は件数から作る |
| DB-08 ストアド内 SQLi | 該当なし | ストアドプロシージャを使っていない |
| DB-09 SQLi からの RCE | 所見あり W-31 | Main の権限は Kosen_map.* だけで FILE は付かない。Logto の Postgres 利用者は SUPERUSER(実測) |
| DB-10 DB ポートの公開 | 問題なし | 本番は 3306・5432 を公開しない。校内 LAN 構成は 3306 を LAN に出す W-26 |
| DB-11 空パスワード・trust | 問題なし | 空パスワード 0 件。pg_hba の trust はコンテナ内の local だけ、host は scram-sha-256(実測) |
| DB-12 アプリが root・postgres で接続 | 所見あり W-31 W-16 | PHP は Main。Logto は SUPERUSER |
| DB-13 DB 接続の非 TLS | 一部 | PHP → MariaDB は CA 必須で TLS。サーバー側の require_secure_transport=0、Logto → Postgres は同じネットワーク内で平文 |
| DB-14 LOCAL INFILE | 所見あり W-08 | PDO 側は既定の無効 |
| DB-15 過剰な GRANT ALL | 所見あり W-09 | — |
| DB-16 バックアップ | 所見あり W-23 | ホストの控えは暗号化し umask 077 |
| DB-17 保存時の暗号化 | 未確認 | ボリュームの暗号化は見ていない |
| DB-18 監査ログ | 未確認 | server_audit・pgaudit の有無は見ていない |
| DB-19 本番データのテスト環境コピー | 未確認 | Old/restore-data.ps1 で LAN へ戻すときに伏せる手順が要るかは読んでいない |
| DB-20 未適用のパッチ | 問題なし(照合なし) | MariaDB 11.4.13・Postgres 17.11(実測)。CVE の照合はしていない |
| DB-21 接続数・タイムアウト | 直した | 指定が無かった。max_connections 120、PDO の ATTR_TIMEOUT 5、Web だけ max_statement_time 15(§5.4) |
| DB-22 エラーの返却 | 所見あり W-20 | — |

### 2.13 Logto(LG)

本番の Logto を読むだけで確かめた(§4)。**カタログの CVE 番号・修正版は一次情報で確かめていない(§0.3)。**

| ID | 結果 | 根拠・備考 |
|---|---|---|
| LG-01〜LG-06 VU#492466(SSO) | 該当なし | **SSO・ソーシャルの接続は 0 件(single_sign_on_enabled=f)、コネクタは SMTP だけ(実測)。** SSO を足す前に、修正版の有無を一次情報で確かめ直すこと |
| LG-07〜LG-09 SAML・XSS・TOTP のリプレイ | 該当なし | 本番は Logto 1.43.0。カタログの修正版 1.41.0 より新しい(**CVE 番号と修正版は一次情報で未確認**)。SAML アプリも作っていない |
| LG-10 logto-tunnel | 該当なし | 使っていない |

### 2.14 ログインフォーム一般(LF)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| LF-01 総当たり | 所見あり W-01 | default テナントは 5 回/60 秒のロック(実測)。admin テナントは sentinel {} |
| LF-02 クレデンシャルスタッフィング | 所見あり W-01 | default テナントは 12 文字以上・漏えい照合・MFA 必須、2 人とも登録済み(実測)。admin テナントは方針なし |
| LF-03 利用者の列挙 | 未確認 | Logto の画面の応答差は試していない |
| LF-04 セッション固定 | 問題なし | callback.php と map-unlock.php で session_regenerate_id(true) |
| LF-05 セッション ID | 所見あり W-33 | use_strict_mode が既定の 0 だった |
| LF-06 Cookie の属性 | 問題なし | secure・httponly・SameSite=Lax |
| LF-07 ログアウト時の失効 | 所見あり W-14 A-05 A-07 | Web は end_session へ送る |
| LF-08 リセットトークン | 該当なし | Logto に任せている |
| LF-09 Host ヘッダ汚染 | 問題なし | HTTP_HOST を使わず APP_URL から組む。Host を偽っても戻り先は変わらない(実測) |
| LF-10 MFA の迂回 | 未確認 | Logto の内部は見ていない。default テナントは必須 |
| LF-11 OAuth の state | 問題なし | PHP SDK と Android SDK が照合 |
| LF-12 redirect_uri | 問題なし | アプリ側は戻り先を絞る。Logto に登録されているのは完全一致の URI(実測) |
| LF-13 PKCE | 問題なし | Android SDK が code_verifier を作る。PHP SDK は実装の知識による |
| LF-14 長期のトークン | 所見あり A-08 | refresh_token の回転と絶対期限(Logto の設定)は見ていない |
| LF-15 クライアント側だけの検証 | 問題なし | パスワード変更と削除は先に /verifications/password |
| LF-16 キャッシュ経由の漏えい | 問題なし | API は no-store、public は静的資材だけ |
| LF-17 CAPTCHA | 所見あり W-32 | ホスト名を照合していなかった |
| LF-18 ログインの監査 | 所見あり W-22 | Logto 側の監査ログはあるが通知は無い |

### 2.15 phpMyAdmin(PM)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| PM-01〜PM-07 既知の CVE | 該当なし | 5.2.3(実測)。カタログの 5.2.2 以降を満たす |
| PM-08 インターネットからの到達 | 所見あり W-03 | IP 制限とゲートの内側 |
| PM-09 root ログイン | 所見あり W-26 | AllowRoot 未指定(=許可)、AllowNoPassword=true だった(実測) |
| PM-10 前段の認証 | 問題なし | nginx の IP 制限 + Logto のゲート(MFA) |
| PM-11 setup/ の残置 | 問題なし | 無い(実測) |
| PM-12 blowfish_secret | 問題なし | 設定されている(実測。長さは見ていない) |
| PM-13 版の露出 | 未確認 | ゲートの内側なので見ていない |
| PM-14 セッションの長さ | 所見あり W-26 | 1800 秒にした |
| PM-15 2FA | 一部 | phpMyAdmin 自体の 2FA は無い。前段のゲートで MFA |
| PM-16 エクスポートの制限 | 未対応 | 制限していない(ゲートの内側) |
| PM-17 HTTP での提供 | 問題なし | 8281 は TLS |

### 2.16 Logto Console・管理系 API(AD)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| AD-01 Console の公開 | 所見あり W-03 | 3002 は IP 制限とゲートの内側 |
| AD-02 Management API のトークン | 問題なし(一部未確認) | M2M の秘密は .env から web にだけ渡す。M2M ロールの範囲は見ていない |
| AD-03 管理者ロールの過剰付与 | 所見あり W-36 A-24 | — |
| AD-04 管理者の MFA | 所見あり W-01 | — |
| AD-05 管理操作の監査 | 所見あり W-22 | 実行者は guard が組んだ利用者から取る |
| AD-06 管理画面の格納型 XSS | 問題なし(Web 側) | 管理画面の出力は km_e、チャットは escapeHtml。Logto Console の中は見ていない |
| AD-07 管理画面の CSRF | 問題なし | すべての POST が km_csrf_verify |
| AD-08 シード管理者 | 未確認 | admin テナントの利用者は 1 人(実測)。初期の管理者かは見ていない |
| AD-09 管理画面と一般画面の同一オリジン | 所見あり W-34 | — |

### 2.17 Docker(DK)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| DK-01 docker.sock のマウント | 問題なし | 無い(実測) |
| DK-02 privileged | 問題なし | 無い(実測) |
| DK-03 root 実行 | 所見あり W-04 W-19 | Soketi を node に。no-new-privileges を足した |
| DK-04 UFW の迂回 | 所見あり W-03 | ufw は有効だが publish した口は迂回する。本番は 3306 を出さず、管理系は nginx の IP 制限が守り |
| DK-05 イメージ内の秘密 | 問題なし | Dockerfile に秘密の ENV・ARG は無い |
| DK-06 イメージのスキャン | 未確認 | Trivy・Grype を走らせていない |
| DK-07 latest タグ | 所見あり W-18 | — |
| DK-08 資源の上限 | 所見あり W-19 W-12 | 直した |
| DK-09 同じネットワーク | 所見あり W-16 | 未対応(certbot だけ別ネットワーク) |
| DK-10 読み取り専用のルート | 所見あり W-19 | 未対応 |
| DK-11 host ネットワーク | 問題なし | 使っていない |
| DK-12 seccomp・AppArmor の解除 | 問題なし | 解除していない |
| DK-13 ログドライバ | 問題なし | json-file 10m × 3(実測) |

### 2.18 nginx(NG)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| NG-01 alias のパストラバーサル | 問題なし | alias は使っていない |
| NG-02 proxy のヘッダ | 所見あり W-21 | web へは X-Real-IP を $remote_addr で上書き。Logto へは利用者の XFF に足していた |
| NG-03 スマグリング | 問題なし | WA-68 と同じ |
| NG-04 autoindex | 問題なし | 指定なし |
| NG-05 レート制限 | 所見あり W-13 W-38 W-27 | — |
| NG-06 弱い TLS | 問題なし | 1.2/1.3 のみ(実測)。OCSP stapling は見ていない |
| NG-07 HSTS | 所見あり W-07 | 443 は max-age=31536000(実測)。Logto の includeSubDomains 付きと二重だった |
| NG-08 server_tokens | 問題なし | off |
| NG-09 ドットファイル | 問題なし | include 専用・設定・vendor などを 404。src にドットファイルは無い |
| NG-10 大きな本文上限 | 問題なし | 3m・12m・210m(配布ファイルの画面だけ) |

### 2.19 Mailpit(MP)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| MP-01 UI の無認証公開 | 所見あり W-25 W-03 | 8025 は IP 制限とゲートの内側 |
| MP-02 SMTP の公開 | 問題なし | 1025 は expose だけ |
| MP-03 本番での稼働 | 所見あり W-25 | 本番でも動いていた(送信は mailserver、Mailpit は 0 通、実測) |
| MP-04 保存期間 | 所見あり W-25 | 本番で止めれば無くなる |

### 2.20 Soketi(SK)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| SK-01 既定の鍵 | 問題なし | 既定値ではない(実測) |
| SK-02 プライベートチャンネルの認証 | 問題なし | chat-auth.php は POST・CSRF・許可リスト・user_id は利用者の sub |
| SK-03 WSS | 問題なし | 6001 は nginx の TLS |
| SK-04 Origin の検査 | 所見あり W-38 | Soketi 自体に Origin を env で指定する口は無い(実測)ので nginx で照合 |
| SK-05 メトリクスの公開 | 問題なし | /usage と /metrics は外から 404(実測) |
| SK-06 接続数の上限 | 所見あり W-04 W-38 | 直した |

### 2.21 OS・ホスト(OS)

| ID | 結果 | 根拠・備考 |
|---|---|---|
| OS-01 SSH のパスワード認証 | 問題なし | PasswordAuthentication no(実測) |
| OS-02 SSH の root ログイン | 問題なし | PermitRootLogin no、AllowUsers km ubuntu(実測) |
| OS-03 自動のセキュリティ更新 | 問題なし | unattended-upgrades が active(実測) |
| OS-04 ファイアウォール | 問題なし | ufw が有効(実測)。DK-04 のとおり publish した口は別 |
| OS-05 fail2ban | 所見あり W-06 | 入っているが failed |
| OS-06 不要なサービス | 所見あり W-25 | コンテナの Mailpit。ホストのほかのサービスは見ていない |
| OS-07 監査ログの外部転送 | 未対応 | 転送していない |
| OS-08 時刻同期 | 問題なし | timesyncd で同期済み(実測) |

## 3. 所見の一覧

**深刻度の順、同じ深刻度の中は観点の順。** 番号の W は Web(server/ とホスト)、A は Android(Test/)。
「場所」は相対パス:行(診断の時点、直す前)。H 系(本番で確認)は場所をホスト上の名前で書いた。

- **反証**の1件(A の最後)は、検証で「既知の未使用ファイルを報告し直しただけ」と分かったもの。直す対象からは外したが、ファイルの扱いは §7 に残した
- **未検証**の6件は、最後の見直し役が足した指摘。うち Postgres の SUPERUSER は本番の実測で裏が取れた。残りは直せるものを直し、確度は低いまま載せている
- **可能性**の件は、備考の「決め手」を確かめるまで確定しない。直せたものは直した

### 3.1 Web(server/ と本番ホスト)

| 番号 | 深刻度 | 判定 | 場所 | 内容 | 直したか | 備考 |
|---|---|---|---|---|---|---|
| W-01 | 中 | 本番で確認 | `Logto の admin テナント(sign_in_experiences)` | Logto Console(admin テナント)の MFA が NoPrompt、パスワード方針 {}、総当たりロック {}。利用者 1 人、MFA 未登録 | **一部** | host-security-check.sh が NoPrompt を知らせる。切り替えは §6 の 23、進め方は §7。パスワード方針と総当たりロックは未対応 |
| W-02 | 中 | 本番で確認 | `ホスト: src/config/app-map.local.php(codes)` | 配信設定のアクセスコードは slug=TEST1 の1件だけで、maps に TEST1 が無い。今のコードではアプリが地図を取れない(500) | **直した(未配備)** | 配信先の無いコードを警告つきで一覧に出し、付け替え・止めるを付けた。**本番での付け替えは §6 の 21** |
| W-03 | 中 | 可能性 | `server/nginx/km/allow-admin.conf:84` | 管理系ポートの IP 制限が 172.16.0.0/12 を許可しており、IPv6 経由(docker-proxy)で外から素通りしうる | **一部** | 管理系 3 ポートを 0.0.0.0 に限定。本番にはグローバル IPv6 が無い(実測)。allow 172.16.0.0/12 は SSH トンネル用に残した |
| W-04 | 中 | 確認済み | `server/docker/soketi/Dockerfile:3` | Soketi が EOL の Node.js 16・bullseye の上で root 実行され、インターネットに公開された 6001 に接続数の制限が無い | **一部** | USER node、接続数・イベント数の上限、nginx の limit_conn。**Node 16 のベースは残る(§7)** |
| W-05 | 中 | 確認済み | `server/src/lib/map-rate-limit.php:34` | アクセスコードの失敗ロックが IP 単位で、会場の共有回線では全来場者の地図取得を15分止められる(読んだ QR はそのまま送信) | **直した(未配備・未配布)** | 'app' の上限を 30 回/15 分、正しいコードは lookup で先に当たりロック中でも通る、IPv6 は /64。アプリは QR を読んでも自動送信しない |
| W-06 | 低 | 本番で確認 | `ホスト: fail2ban.service` | fail2ban が起動直後に exit 255 で止まったまま(failed)。/etc/fail2ban/jail.local はある。プロジェクトのスクリプトはこれを見ていなかった | **一部** | host-security-check.sh が「入っているのに動いていない」を知らせる。起動は §6 の 22 |
| W-07 | 低 | 本番で確認 | `server/nginx/default.conf.template(3001・3002 の server)` | Logto が自分で includeSubDomains 付きの HSTS を送り、nginx の HSTS と二重になっていた | **直した(未配備)** | proxy_hide_header Strict-Transport-Security |
| W-08 | 低 | 本番で確認 | `server/compose.yaml(mariadb の command)` | MariaDB の local_infile=1(アプリは LOAD DATA を使っていない) | **直した(未配備)** | --local-infile=0 |
| W-09 | 低 | 本番で確認 | `MariaDB の mysql.user` | root@% がある。アプリの Main@% は GRANT ALL PRIVILEGES ON Kosen_map.*(全体の FILE は無い) | **利用者の判断待ち** | §7 |
| W-10 | 低 | 本番で確認 | `server/src/admin/map-publish.php` | 管理画面の CSP(nonce 方式)の下で、map-publish.php の onsubmit= と nonce 無しの <script> が動いていなかった | **直した(未配備)** | data-km-confirm と nonce 付きの <script> |
| W-11 | 低 | 本番で確認 | `ホスト: /etc/letsencrypt/renewal/ito8795.com.conf` | 証明書の更新方式が standalone のまま。コンテナのループは --webroot を渡すので通るが、素の certbot renew は失敗する | **直した(未配備)** | host-cert.sh fix-conf。**本番での実行は §6 の 9** |
| W-12 | 低 | 本番で確認 | `web コンテナ(Apache)とすべてのコンテナ` | Apache の Timeout 300。コンテナに mem/pids の上限も no-new-privileges も無い | **直した(未配備)** | Apache Timeout 60、MaxRequestWorkers 30(web の mem_limit 384m に合わせて)。全コンテナに mem_limit・pids_limit・no-new-privileges(mailserver を除く) |
| W-13 | 低 | 確認済み | `server/nginx/default.conf.template:272` | nginx の完全一致 location が PATH_INFO 付きの URL で素通りされる(レート制限と include 専用ファイルの遮断が外れる) | **直した(未配備)** | nginx の 443 の最初の正規表現で `\.php/` を 404。include 専用の遮断も `.php/…` を含める。Apache に AcceptPathInfo Off。**本番では /contact.php/x・/logto-client.php/x が 200 だった(実測で成立を確認)** |
| W-14 | 低 | 可能性 | `server/src/account.php:42` | account.php だけが停止判定をしていない —— 停止済みの利用者が既存セッションで表示名の変更・アカウント削除(監査ログの匿名化)まで進める可能性 | **直した(未配備)** | account.php に停止判定。停止中は監査ログに残してセッションを捨てる |
| W-15 | 低 | 確認済み | `server/src/Main/app.js:2002` | 公開地図の経路表示で、ノード名をエスケープせずに Leaflet のツールチップへ渡している(HTML 注入) | **直した(未配備)** | bindTooltip(escapeHtml(node.name))。check.php が素で渡していないか見張る |
| W-16 | 低 | 確認済み | `server/compose.yaml:164` | web コンテナに Logto の DB(Postgres)資格情報を渡しているが PHP は使っていない。しかも全コンテナが同じネットワーク 1 本 | **一部** | web の env から POSTGRES_* を外した。ネットワークの分割は未対応。Postgres の SUPERUSER は §7 |
| W-17 | 低 | 確認済み | `server/compose.yaml:152` | LOGTO_WEBHOOK_SIGNING_KEY が web コンテナに渡っておらず、アカウント削除の後片付けが本番で動かない | **直した(未配備)** | web に LOGTO_WEBHOOK_SIGNING_KEY を渡した。検証失敗は監査ログへ(10 分に 1 件) |
| W-18 | 低 | 確認済み | `server/docker/php/Dockerfile:1` | イメージの固定が digest とタグで混在し、ホスト上の補助コンテナは動くタグのまま | **未対応(約束の範囲外)** | digest 固定はしていない |
| W-19 | 低 | 確認済み | `server/compose.yaml:229` | コンテナの硬化が無い(非 root・capability・読み取り専用・資源上限)。require される PHP 設定ファイルを www-data が書き換えられる | **一部** | mem_limit・pids_limit・no-new-privileges(mailserver を除く)。read_only・cap_drop・src の :ro、config/*.local.php を www-data が書ける点は残る |
| W-20 | 低 | 確認済み | `server/src/contact.php:114` | 例外メッセージを画面にそのまま出す箇所が残っている(公開の問い合わせフォームと多数の管理画面) | **直した(未配備)** | lib/user-error.php(KmUserError・照合用 ID)。contact.php と管理画面の getMessage 表示を置き換え |
| W-21 | 低 | 可能性 | `server/nginx/default.conf.template:403` | Logto(3001/3002)へ利用者が付けた X-Forwarded-For をそのまま足して渡しており、Logto 側で IP を偽装できる | **直した(未配備)** | 3001・3002 は X-Forwarded-For を $remote_addr だけにした |
| W-22 | 低 | 確認済み | `server/src/admin/_inc/guard.php:111` | 監査ログの網羅が不足(拒否・停止・サインアウト・アプリのコード失敗・webhook 失敗が記録されない) | **一部** | admin.denied・admin.write_denied・gate.denied・account.suspended_blocked・logout・webhook.signature_failed を記録。アプリのコード失敗の記録と外部転送は未対応 |
| W-23 | 低 | 確認済み | `server/scripts/open-backup.ps1:107` | 手元の控えの置き場(D:\Backups)は Authenticated Users が変更可能。開いた平文を『24 時間で消える』と表示するが、実際は最長で約 1 週間残る | **一部** | 保持期間の表示を実際に合わせ、削除を Test-Path で確かめる。D:\Backups の ACL は未対応 |
| W-24 | 低 | 確認済み | `server/scripts/host-updates-setup.sh:210` | 配備用の km が実質ホストの root。root の cron が km 所有のスクリプトを実行し、手元 PC の鍵にはパスフレーズが無い | **未対応(約束の範囲外)** | km が docker グループ経由で実質 root である構成は変えていない |
| W-25 | 低 | 確認済み | `server/compose.vps.yaml:94` | 本番でも Mailpit を起動・公開している。メール送信先の既定値も Mailpit | **直した(未配備)** | compose.vps.yaml で mailpit を profiles の内側に。nginx は mailpit が居なくても起動する。**本番で止めて消す(§6 の 6)** |
| W-26 | 低 | 確認済み | `server/compose.yaml:117` | phpMyAdmin と MariaDB の root がイメージの既定のまま(root ログイン許可・2FA なし)。校内 LAN 構成では 3306 も LAN に公開 | **一部** | phpMyAdmin に config.user.inc.php(AllowRoot は KM_PMA_ALLOW_ROOT=1 のときだけ、AllowNoPassword=false、LoginCookieValidity=1800)。root@% は §7。LAN の 3306 は変えていない |
| W-27 | 低 | 確認済み | `server/nginx/default.conf.template:92` | nginx のレート制限が GET を一律に除外し、limit_conn も無い。未認証で重い GET(APK の PHP 配信など)を並列で叩ける | **直した(未配備)** | km_api(240r/m)と limit_conn。APK を X-Accel-Redirect で配るのは範囲外 |
| W-28 | 低 | 確認済み | `server/scripts/host-emergency.sh:63` | ホストの緊急調査の出力が誰でも読める権限で /tmp に残る | **直した(未配備)** | host-emergency.sh に umask 077 |
| W-29 | 低 | 可能性 | `server/nginx/default.conf.template:417` | Logto Console(3002)にフレーム埋め込みの禁止を付けていない | **対応不要** | 本番で Logto 自身が frame-ancestors を送っていることを実測した |
| W-30 | 低 | 未検証 | `server/src/api/app-map.php:64` | 合言葉とアクセスコードの失敗ロックが「判定→照合→記録」の順で原子的でなく、同時に送ると8回の上限を超えて試せる(/api/app-map.php は nginx の回数制限もかかっていない) | **直した(未配備)** | 照合の前に INSERT … ON DUPLICATE KEY UPDATE で数える(km_map_unlock_attempt)。/api/app-map.php に km_appmap |
| W-31 | 低 | 未検証 | `server/compose.yaml:51` | Logto と web コンテナが、Postgres の初期ユーザー(= スーパーユーザー)の資格情報で繋ぐ構成になっている | **利用者の判断待ち** | **本番の実測で SUPERUSER を確認**。非 SUPERUSER への移行は §7 |
| W-32 | 低 | 未検証 | `server/src/lib/recaptcha.php:263` | reCAPTCHA の検証で、トークンが出たホスト名(と action)を照合していない | **直した(未配備)** | hostname を APP_URL のホストと照合。Enterprise は action も |
| W-33 | 低 | 未検証 | `server/src/lib/session.php:31` | PHP のセッション設定が既定のまま(session.use_strict_mode 無効、Cookie 名は PHPSESSID で __Host- 接頭辞なし) | **一部** | use_strict_mode・use_only_cookies・use_trans_sid を ini と session.php の両方で。__Host- 接頭辞は未対応(全員が一度ログアウトされるため) |
| W-34 | 低 | 未検証 | `server/nginx/default.conf.template:307` | 管理画面(/admin/)が公開の地図やフォームと同じオリジンにあり、セッション Cookie も共有している | **未対応(約束の範囲外)** | 管理画面を別オリジンに分けていない |
| W-35 | 低 | 確認済み | `server/src/api/app-map.php:77` | 成功すると 'app' の失敗回数が消えるため、配布済みのコードを混ぜれば無制限に試行でき、1回ごとに全コード分の bcrypt を実行させられる | **直した(未配備)** | 成功しても 'app' の失敗回数を消さない。照合は HMAC の lookup で1回、bcrypt は古いコードだけ(上限 10 件) |
| W-36 | 情報 | 確認済み | `server/src/admin/_inc/bootstrap.php:62` | 管理画面の全権限が1つのスコープ admin:users:read だけで決まっている(書き込み用スコープを見ていない) | **直した(未配備)** | 状態を変える要求とゲートに admin:users:write も要求(KM_ADMIN_WRITE_SCOPE) |
| W-37 | 情報 | 確認済み | `server/src/sign-out.php:14` | サインアウトが GET で CSRF トークンを取らない(ログアウト CSRF) | **直した(未配備)** | sign-out.php は POST + CSRF のときだけ。リンクはフォームに |
| W-38 | 情報 | 確認済み | `server/nginx/default.conf.template:454` | Soketi(6001)に Origin 制限も入口の回数制限も無い | **直した(未配備)** | nginx の 6001 で Origin を KM_APP_URL と照合、limit_conn 50。Soketi に接続数・イベント数の上限 |
| W-39 | 情報 | 確認済み | `server/src/admin/monitor.php:134` | <script> へ埋め込む json_encode に JSON_HEX_TAG が付いていない箇所が残っている(値は設定由来) | **直した(未配備)** | <script> 内の json_encode に JSON_HEX_TAG・AMP・APOS・QUOT |
| W-40 | 情報 | 確認済み | `server/scripts/host-backup.sh:235` | DB の root パスワードを運用スクリプトのコマンド引数に載せている | **直した(未配備)** | MYSQL_PWD で渡す。バックアップ専用の DB 利用者は作っていない |
| W-41 | 情報 | 確認済み | `server/nginx/default.conf.template:327` | OIDC コールバックの認可コードが nginx のアクセスログに残る | **直した(未配備)** | location = /callback.php のアクセスログを km_no_query に |
| W-42 | 情報 | 確認済み | `server/scripts/host-backup.sh:302` | バックアップの暗号が認証付きでない(改ざんを検出できない) | **未対応(約束の範囲外)** | 認証付き暗号(署名)への変更はしていない |
| W-43 | 情報 | 未検証 | `server/nginx/default.conf.template:258` | 静的資材の正規表現 location が、上流の応答に関係なく Cache-Control: public, max-age=30日 を付ける(PATH_INFO 付きの PHP 応答にも付く) | **直した(未配備)** | PATH_INFO 付きの .php を 404 にしたので、静的資材の location に落ちない |

### 3.2 Android(Test/)

| 番号 | 深刻度 | 判定 | 場所 | 内容 | 直したか | 備考 |
|---|---|---|---|---|---|---|
| A-01 | 中 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/RankingRepository.kt:138` | ランキング参加の既定が ON で、ログイン利用者の表示名がログイン不要の公開ランキングに本人の同意なく載る(画面の説明文とサーバーの前提にも反する) | **直した(未配布)** | 既定を OFF、ON は同意画面を通す。同意の記録が無い ON は OFF に戻す。**溜まった行は §7** |
| A-02 | 中 | 可能性 | `Test/app/src/main/java/com/ito/kosenmap/MainActivity.kt:320` | ログアウトの完了時に、止まっている Activity で FragmentTransaction.commit() を呼んでクラッシュする | **直した(未配布)** | 画面の切り替えを lifecycle.withStarted の中へ(実機では未確認) |
| A-03 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/ImportExportManager.kt:100` | 設定の取り込みがキーごとの型を確かめないため、型の違う値が入ると起動時に ClassCastException で落ち続ける | **直した(未配布)** | 設定の型の表(expectedSettingType)で取り込み時に捨て、起動時にも掃除。TypeSafePreferences |
| A-04 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MainActivity.kt:438` | アカウント画像を原寸でデコードしており、サーバーも縦横のサイズを制限していない | **直した(未配備・未配布)** | アプリは inSampleSize で縮小デコード(256px)。サーバーは縦横 2048px を超えたら断る |
| A-05 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MainActivity.kt:314` | ログアウトしても、スタッフ版の地図(教職員氏名)とアカウント画像が端末に残る。氏名は役割に関係なく表示・検索される | **直した(未配布)** | ログアウトでトークン付きの地図から氏名とスタッフ限定の地点を落とす。ログインし直したら取り直す |
| A-06 | 低 | 確認済み | `Test/Test.zip:1` | Test プロジェクト直下の Test.zip に DB 接続設定ファイル(パスワードらしき項目あり)と local.properties が入ったまま残っている | **利用者の判断待ち** | .gitignore に /*.zip を足しただけ。退避と中の DB 資格情報のローテートは §7 |
| A-07 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MainActivity.kt:314` | ログアウト時に OIDC 設定を取得できないと refresh_token の失効(revoke)が行われないが、アプリは成功として扱う | **直した(未配布)** | 失効を確かめられなければ refresh_token を暗号化して保留し、次に通信できたときに送り直す |
| A-08 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/auth/logto/LogtoAuthManager.kt:44` | Logto の refresh_token / id_token が平文の SharedPreferences に保存される | **直した(未配布)** | Android Keystore の AES-256-GCM で包んで保存。既存の平文は一度だけ移す |
| A-09 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/ImportExportManager.kt:105` | 設定の取り込み(ファイル・アカウント復元)で値の型を検査せず、型の違う値が入ると起動のたびに ClassCastException で落ちる | **直した(未配布)** | A「設定の取り込みの型」と同じ修正 |
| A-10 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/AdMapScreen.kt:267` | 管理マップを開いただけで「この端末で直した」印が立ち、設定画面が「まだ Website に渡っていません」と誤表示する | **直した(未配布)** | 内容が変わらなければ保存も「直した」印も立てない |
| A-11 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/AdMapScreen.kt:271` | 管理マップの自動保存(300ms 待ち)が画面を離れると取り消され、その後の設定画面のイベント操作で未保存の編集が消える | **直した(未配布)** | 離れるときに即保存。設定画面のイベント操作は ViewModel 経由 |
| A-12 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MapScreen.kt:161` | イベントの会期の境目(開始・終了)が、地図を開いたままだと反映されない | **直した(未配布)** | 次の会期の境目で計算し直す(nextEventBoundaryMillis) |
| A-13 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MapScreen.kt:314` | 地図画面に戻るたびに表示中の階が既定の階に戻り、平滑化と階の安定化もリセットされる | **直した(未配布)** | 既定の階は値が変わったときだけ適用 |
| A-14 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/RankingScreen.kt:168` | ランキングの順位を rows.indexOf(row) で出しているため、内容が同じ行に同じ(誤った)順位が付く | **直した(未配布)** | itemsIndexed の index で順位 |
| A-15 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MapQrScanner.kt:185` | QR 読み取りダイアログをすぐ閉じると、カメラが解放されずに動き続けることがある | **直した(未配布)** | disposed の印。閉じた後のカメラの準備では bind しない |
| A-16 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/LocationDisclosure.kt:127` | 位置情報の開示ダイアログが画面の回転で消えると、その起動中は権限を二度と求めない(管理マップでは理由も出ない) | **直した(未配布)** | 答えを受け取ったときに印を立てる。管理マップにも理由の表示 |
| A-17 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/WifiScanController.kt:77` | Wi-Fi スキャンの要求が制限されたとき、古いスキャン結果を新しい観測として移動平均に積み直す | **直した(未配布)** | 測位の画面では制限時に古い結果を積まない |
| A-18 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/WifiViewModel.kt:401` | 配信マップの適用・読み直し・起動時の読み込み、およびファイルの書き出し/取り込みを、メインスレッドで行っている | **直した(未配布)** | 読み込み・デコード・保存・SAF を IO/Default へ。世代番号で古い結果の上書きを防ぐ |
| A-19 | 低 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MainActivity.kt:447` | ログアウトや利用者の切り替えの直後に、取得中だった前の利用者のアカウント画像が表示されることがある | **直した(未配布)** | 取得の Job を取り消し、反映の直前に同じ利用者かを確かめる |
| A-20 | 情報 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/WifiViewModel.kt:283` | 別の配信(mapId)へ切り替えるときも版番号だけを比べるので、版の小さい地図に切り替えられない | **直した(未配備・未配布)** | haveMapId を送り、サーバーは配信 ID ごとに版を比べる。mapId が違えば置き換え対象(上書き確認を通す) |
| A-21 | 情報 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MapDownloadClient.kt:68` | サーバー応答・取り込みファイルをサイズの上限なしで全部メモリ(またはキャッシュ)に読み込む | **直した(未配布)** | HttpLimits.kt: 地図 20MB・JSON 1MB・画像 2MB。Content-Length と読みながらの両方で数える |
| A-22 | 情報 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/AccountRepository.kt:55` | アバターのキャッシュがサインアウト後も残り、ファイル名が sub の 32bit hashCode | **直した(未配布)** | キャッシュ名を sub の SHA-256 の先頭 128bit に。ログアウトで全部消す |
| A-23 | 情報 | 確認済み | `Test/app/build (1).gradle.kts:38` | 古い app/build (1).gradle.kts が残っている(校内 LAN の接続先、minify 無効) | **利用者の判断待ち** | A「build (1).gradle.kts」と同じ件(§7) |
| A-24 | 情報 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/auth/logto/LogtoAuthManager.kt:42` | 来場者ビルドも admin:* 権限(users:write・api-keys:write を含む)を要求しており、最小権限になっていない | **直した(未配布)** | 来場者ビルドは admin:* を要求しない |
| A-25 | 情報 | 確認済み | `Test/app/build.gradle.kts:225` | 本番の認証に Logto Android SDK のベータ版(3.0.0-beta)を使っている | **未対応(安定版が無い)** | build.gradle.kts にコメントを足しただけ |
| A-26 | 情報 | 確認済み | `Test/app/build (1).gradle.kts:66` | 旧設定の build (1).gradle.kts が残っている(LAN の IP、minify 無効、KSP・Room・Coil など古い依存) | **利用者の判断待ち** | 動かしていない(§7) |
| A-27 | 情報 | 確認済み | `Test/app/src/main/java/com/ito/kosenmap/MapScreen.kt:296` | 地図画面のセンサー(回転ベクトル・気圧・加速度・歩数)は、アプリが裏に回っても登録されたまま | **直した(未配布)** | STARTED 以上の間だけ登録 |
| A-28 | 情報 | 反証 | `Test/app/src/main/java/com/ito/kosenmap/MapLayerRenderer.kt:10` | 未使用のまま残っているファイル(MapLayerRenderer.kt / NodeInfoDialog.kt / file_paths.xml) | **反証(扱いは判断待ち)** | 既知の未使用ファイルを報告し直しただけ。4 ファイルの扱いは §7 |

### 3.3 「可能性」の件の決め手

| 番号 | 何を確かめれば確定するか |
|---|---|
| W-14 | 停止した利用者の発行済みトークンで、Logto の Account API(/verifications/password・/my-account)が通るか。通らなければ成り立たない |
| W-03 | VPS に IPv6 アドレスがあるか(**本番に無いと実測**)。devnet の中の侵害されたコンテナから 172.x として届く経路は残る |
| W-21 | X-Forwarded-For を付けてサインインを1回失敗させ、Logto Console の監査ログに出る IP を見る |
| W-29 | 3002 の応答に frame-ancestors があるか(**本番で Logto 自身が送ると実測 → 対応不要**) |
| A-02 | 実機でログアウトしても落ちないか(直した上で、実機では未確認) |

## 4. 本番ホストの実測(2026-09-14、読むだけ)

**値を読んだだけで、何も変えていない。秘密の値はここに書かない。**

| 項目 | 実測 | 関係 |
|---|---|---|
| OS・資源 | Ubuntu 24.04.4 / kernel 6.8.0-139 / 2 CPU / RAM 1967MB(使用 1148MB)/ Docker 29.7.2 / compose 5.5.0 | 負荷の値の前提 |
| compose の既定値 | 入れ子の既定値 `${A:-https://${KM_DOMAIN}}` が効く | ドメインの導出(§5.2) |
| .env | COMPOSE_FILE="compose.yaml:compose.vps.yaml"。URL 系は全部明示で入っている。MAIL_HOST=mailserver、web の MAIL_PORT は 587。LOGTO_WEBHOOK_SIGNING_KEY は .env にあるが web に渡っていなかった | W-17、§5.2 |
| 配備の状態 | 手元の server/ と本番の主要ファイルは一致(この報告の修正より前の分まで) | — |
| 公開ポート | 80/443/3001/3002/6001/8025/8281(IPv4 と [::] の両方を docker-proxy が待つ)。**グローバル IPv6 は無い** | W-03 |
| ファイアウォール・SSH | ufw 有効 / PermitRootLogin no / PasswordAuthentication no / AllowUsers km ubuntu | OS-01・02・04 |
| 更新・時刻 | unattended-upgrades active / timesyncd active(同期済み) | OS-03・08 |
| fail2ban | **failed**(04:55 の起動直後に exit 255)。jail.local はある | W-06 |
| コンテナ | privileged なし、docker.sock のマウントなし、ログは json-file 10m×3。**mem/pids の上限なし、no-new-privileges なし**。certbot だけ別ネットワーク | W-19 W-12 |
| メモリの使用 | logto 254M / mariadb 133M / web 58M / soketi 54M / mailserver 54M / phpmyadmin 51M / postgres 45M / mailpit 23M / nginx 10M / certbot 5M | mem_limit の値の根拠 |
| web コンテナ | PHP 8.4.25 / Apache 2.4.68 / Debian 13。Apache Timeout 300、KeepAliveTimeout 5、MaxRequestWorkers 150。env に POSTGRES_* があった(PHP は使っていない) | W-16 W-12 |
| www-data の書き込み | docroot・lib・api は書けない。config/db.local.php と config/map-access.local.php(www-data:km 640)は書ける。uploads・cache は書ける(必要) | W-19 |
| PATH_INFO | `/contact.php/x` 200、`/logto-client.php/x` 200、`/admin/index.php/a.css` に public, max-age=2592000 | W-13 |
| MariaDB 11.4.13 | local_infile=1、require_secure_transport=0、max_connections=151、max_statement_time=0、wait_timeout=28800。利用者: root@localhost・**root@%**・healthcheck・**Main@%(ALL ON Kosen_map.*)**。空パスワード 0 件。LOAD DATA・INTO OUTFILE は不使用 | W-08 W-09 |
| Postgres 17.11 | Logto の接続利用者は **SUPERUSER**。pg_hba は local trust(コンテナ内)+ host scram-sha-256 | W-31 |
| Logto 1.43.0 | コネクタは SMTP だけ。**SSO・ソーシャルの接続 0 件**。logto-tunnel なし | LG-01〜10 |
| Logto default テナント | MFA Mandatory(Totp/WebAuthn/EmailVerificationCode)、パスワード 12 文字以上 + 漏えい照合、ロック 5 回/60 秒。利用者 2 人とも MFA 登録済み | LF-01・02 |
| Logto admin テナント(Console) | **MFA NoPrompt、password_policy {}、sentinel {}**。利用者 1 人、MFA 未登録 | W-01 |
| Logto のアプリ | KosenAPP(Native、来場者と管理の2つのリダイレクト URI)、Test(Traditional、https://ito8795.com/callback.php)、Kosen_map(M2M) | LF-12 |
| Logto のヘッダー | 3001/3002 で自分で frame-ancestors と HSTS(includeSubDomains 付き)を送る → nginx の HSTS と二重 | W-07 W-29 |
| phpMyAdmin 5.2.3 | setup/ なし、blowfish_secret あり、AllowNoPassword=true(イメージの既定)、AllowRoot 未指定(=許可) | W-26 |
| Soketi | Node v16.20.2、soketi 1.6.1、root 実行。鍵は既定値でない。上限の env は受け付けるが、許可 Origin を指定する口は無い。外から / と /ready は 200、/usage・/metrics は 404、任意の Origin で 101 | W-04 W-38 |
| Mailpit v1.31.0 | 本番でも動いている(0 通)。送信は mailserver | W-25 |
| HSTS・TLS | 443 は max-age=31536000。TLS 1.0/1.1 は拒否 | NG-06・07 |
| CORS・Host | CORS ヘッダーは返さない。Host を偽っても戻り先は APP_URL 固定 | WA-06、LF-09 |
| 証明書 | /etc/letsencrypt/live/ito8795.com(ECDSA、SAN は ito8795.com のみ、期限 2026-11-26)。**renewal の authenticator = standalone**。`host-cert.sh renew --dry-run` は成功 | W-11 |
| 定期実行 | /etc/cron.d/kosenmap-updates は版 5(backup・check-updates・host-security-check) | §6 の 8 |
| 地図の配信設定 | maps は kosen-main だけ(revision 1、期限 2026-10-14)。**codes は slug=TEST1 の1件だけ(maps に無い)、平文の控えなし** | W-02 |
| 管理画面の CSP | script-src 'self' 'nonce-…' のみ → map-publish.php の onsubmit= と nonce 無しの script は動かない | W-10 |

## 5. 直したこと(改修の結果)

**どれも手元だけ。本番には入っていない。** 担当ごとに実装し、別の担当が差分を全部読んで見直した。
見直しで見つかった「直したことによる新しい穴」も、ここで直してある(例: 未認証の人がサインアウトを繰り返すと監査ログを埋められた)。

### 5.0 手元で確かめたこと(全体)

| 何を | 結果 |
|---|---|
| `php src/scripts/check.php`(server で、引数なし) | **すべて通過(951 件)**。着手前は 823 件で、今回の守りが戻らないよう `hardening` の節を足した |
| 変えた PHP の構文(`php -l`) | すべて構文エラーなし |
| 変えた JS(`node --check`)と既存の JS テスト 3 本 | すべて exit 0 |
| Android(visitor と admin の両フレーバー) | **単体テスト 571 件、失敗 0**。両方の APK を作った |
| 新しい compose | 本番の .env の形で `docker compose config` が通る。**KM_DOMAIN だけの .env でも URL が導かれる** |
| nginx | 本番と同じイメージ・証明書で `nginx -t` が通る |
| Apache・PHP | Syntax OK、AcceptPathInfo Off、Timeout 60。PHP は use_strict_mode=1、default_socket_timeout=15 |
| Soketi | node 利用者で起動する |
| 文字コード | .php・.js・.sh・.kt は BOM なし LF、.ps1 は BOM 付き、ja.js・en.js は元の CRLF のまま |
| **実機・ブラウザ** | **見ていない**(§8) |

### 5.1 セキュリティ

| 場所 | 何を変えたか | 所見 |
|---|---|---|
| nginx | 443 の最初の正規表現で `.php/…` を 404。include 専用ファイルの遮断も PATH_INFO 付きを含める | PATH_INFO |
| Apache | `docker/php/apache-km.conf`(AcceptPathInfo Off・Timeout 60)を compose.yaml と compose.vps.yaml の両方にマウント | 同上 |
| nginx | 3001・3002 は X-Forwarded-For を $remote_addr だけ、Logto の HSTS を隠す。443 に Permissions-Policy(geolocation=() など。公開地図も管理画面も位置情報を使わないことを grep で確認)。/callback.php のログにクエリを残さない。6001 で Origin を KM_APP_URL と照合 | XFF・HSTS 二重・認可コード・CSWSH |
| compose | web の env から POSTGRES_* を外し、LOGTO_WEBHOOK_SIGNING_KEY を渡す。VPS の管理系 3 ポートを 0.0.0.0 に。**本番では Mailpit を起動しない**(profiles)、nginx は Mailpit が居なくても起動する | 資格情報・webhook・Mailpit |
| phpMyAdmin | `docker/phpmyadmin/config.user.inc.php`: AllowRoot は KM_PMA_ALLOW_ROOT=1 のときだけ、AllowNoPassword=false、LoginCookieValidity=1800 | root ログイン |
| MariaDB・コンテナ | `--local-infile=0`。mailserver 以外に no-new-privileges(postfix の postdrop が setgid を使うため mailserver は除く) | local_infile・硬化 |
| PHP | `lib/user-error.php`(KmUserError と照合用 ID)で例外の文面を画面に出さない。account.php の停止判定。サインアウトは POST + CSRF。状態を変える要求とゲートに admin:users:write。監査ログの追加。<script> 内の json_encode を HEX で逃がす。reCAPTCHA のホスト名(と action)照合。session の strict mode。管理画面のインラインの on…= と nonce 無しの <script> を無くした | 例外・停止・CSRF・スコープ・監査・CAPTCHA・セッション |
| JS | 公開地図の経路のツールチップを escapeHtml | DOM XSS |
| scripts | host-security-check.sh が「fail2ban が入っているのに動いていない」「Logto Console の MFA が Mandatory でない」を知らせる。host-emergency.sh に umask 077。host-backup.sh の DB パスワードを MYSQL_PWD で渡す。open-backup.ps1 の保持期間の表示と削除の確認 | fail2ban・MFA・/tmp・引数・控え |
| Android | ランキングは同意してから(既定 OFF)。トークンを Keystore(AES-256-GCM)で保存。失効できなければ保留して送り直す。来場者ビルドは admin:* を要求しない。ログアウトでスタッフの情報を落とす。アバターのキャッシュ名を SHA-256 に | Android の所見 |

**手元で確かめたこと:** check.php の `hardening` が上の各項目を文字列と関数の呼び出しで見張る(反例 16 通りで検査が壊れた入力を捕まえることも確かめた)。
KmUserError・km_user_error_message・reCAPTCHA のホスト名照合は CLI の小さな試験で動かした。Android は SecurityHardeningTest 14 件・SecureTokenStorageTest 5 件・RankingConsentTest 6 件。
**ブラウザでの見た目(ログアウトのボタン、確認ダイアログが1回だけ出るか、write スコープの 403)は見ていない。**

### 5.2 ドメインを .env で変える

**本番の .env で必須なのは `KM_DOMAIN` だけになった。** compose.vps.yaml が次を導く(.env に書けばそちらが勝つ):

| 値 | 既定 |
|---|---|
| KM_APP_URL・APP_URL | `https://${KM_DOMAIN}` |
| KM_APP_PORT | 443 |
| KM_CERT・KM_CERT_KEY | `/etc/letsencrypt/live/${KM_DOMAIN}/fullchain.pem`・`privkey.pem` |
| LOGTO_ENDPOINT・LOGTO_ADMIN_ENDPOINT | `https://${KM_DOMAIN}:3001`・`:3002` |
| PMA_ABSOLUTE_URI | `https://${KM_DOMAIN}:8281/` |
| MAIL_FROM・MAIL_HOST・MAIL_PORT | `noreply@${KM_DOMAIN}`・mailserver・587 |
| mailserver の ALLOWED_SENDER_DOMAINS・myhostname | KM_DOMAIN から |

- VPS で KM_DOMAIN が無いと、`${KM_DOMAIN:?…}` で**起動の前に止まる**。校内 LAN(compose.yaml)の既定値は変えていない
- **新しい env `KM_API_RESOURCE`** = Logto の API リソース識別子(アクセストークンの audience)。**ドメインを変えても変えない。** 変えると配布済みのアプリのトークンが全部通らなくなる。web に LOGTO_API_RESOURCE と KOSENMAP_LOGTO_AUDIENCE として渡す(既定は VPS が `https://${KM_DOMAIN}/api`、LAN が `https://192.168.3.29:9443/api`)
- **`scripts/host-domain.sh`**: `check NEW` は何が変わるかを見るだけ。`apply NEW` は .env を `.env.bak-<日時>`(600)に控え、KM_DOMAIN を書き換え、旧ドメインを含む URL 系の行を**消さずにコメントにして**導出に任せ、KM_API_RESOURCE の行が無ければ今の audience を書いて固定する。**新ドメインの証明書が無ければ書き換える前に止まる。** 書き換えた写しで compose を読ませ、APP_URL と LOGTO_ENDPOINT が新ドメインを指すか確かめてから置き換える。**自分では up しない**
- **`src/scripts/logto-domain.php`**: Logto のリダイレクト URI・サインアウト後の URI・CORS・webhook などのうち、**ホスト名が旧ドメインと完全に一致するものだけ**を新ドメインに換える。既定は一覧だけ、`--apply` で書き換えて読み直す
- **Android**: `-Pkosenmap.domain=<ドメイン>`(既定 ito8795.com)で siteUrl と logtoEndpoint を導く。個別の上書きは `kosenmap.siteUrl`・`kosenmap.logtoEndpoint`・`kosenmap.apiResource`(既定 `https://ito8795.com/api`、**ドメインからは導かない**)。読む順は -P → gradle.properties → 環境変数 KOSENMAP_DOMAIN。形が不正か https 以外なら設定の段階で止まる

**ドメインを変えるときの順番**(今回は実行しない。実行できるセルは [09-new-host](09-new-host.ipynb) の §6「ドメインを変える」):
DNS の A レコード → `host-cert.sh issue --domain NEW --email …` → `host-domain.sh check NEW` → `host-domain.sh apply NEW` → `docker compose up -d` →
`docker compose exec -T -u www-data web php scripts/logto-domain.php --from=ito8795.com`(一覧)→ 同じコマンドに `--apply` → DKIM・SPF・DMARC・PTR、reCAPTCHA のドメイン、Android の両フレーバーを作り直す。

**手元で確かめたこと:** compose の併合を再現する検査で、KM_DOMAIN が無いと止まる・`example.test` なら全部がそのドメインを指す・LAN の既定は変わらないことを確かめた。本番の .env の形で `docker compose config` が通り、KM_DOMAIN だけの .env でも URL が導かれる。
host-domain.sh の書き換え部分(awk)を4通りの .env で、ホスト名の照合を7通りで試した(`kosen.ito8795.com` のような旧の一部を新ドメインと誤認しない)。Ctrl+C で .env が空になる経路は見直しで塞いだ。
logto-domain.php の重複のまとめ方を3通りで試した。Android は生成された BuildConfig の値と、不正なドメインで止まることを確かめた。
**host-domain.sh を丸ごと走らせる試験はできていない**(この PC の sh が fork に失敗する)。

### 5.3 Let's Encrypt の更新

**`scripts/host-cert.sh`**(本番用):

| コマンド | 何をするか |
|---|---|
| `status` | 残り日数と、nginx が**実際に出している**証明書(読むだけ) |
| `renew` | 期限が近ければ更新し、更新されたら nginx を reload して、出している証明書の指紋まで突き合わせる |
| `renew --dry-run` | 試験用の発行元で手順だけ通す |
| `issue --domain D --email E` | 新しいドメインの証明書を取る(.env と nginx は変えない) |
| `fix-conf [--dry-run]` | renewal/*.conf の authenticator を webroot に揃える |

**なぜ要るか:** certbot コンテナのループは失敗しても誰にも知らせず、更新しても nginx は 12 時間ごとの reload まで古い証明書を出し続ける。HSTS を有効にしてあるので、**切れた時点で誰もサイトに入れなくなる。**

**cron を版 6 に**(host-updates-setup.sh): 毎日 03:47 に `send-log.sh --only-failure` 経由で `host-cert.sh renew`(失敗したときだけメール)、毎月 1 日 04:07 に `status` を必ず送る(届かなければ仕掛けが止まっている)。
host-setup.sh の HOST_SCRIPTS と deploy-to-host.ps1 の送る一覧・必須の一覧に host-cert.sh と host-domain.sh を足した。

**手元・本番で確かめたこと:** 本番で `status` と `renew --dry-run` が成功した(読むだけ・試験用の発行元)。check.php が cron の版 6・時刻・`--only-failure`・実行ビットの対象を見張る。
**版 6 の cron と fix-conf は、まだ本番で実行していない(§6 の 8・9)。**

### 5.4 負荷(意図的に重くされる場所)

| 場所 | 何を変えたか |
|---|---|
| nginx の待ち | client_header_timeout 15s、client_body_timeout 30s、send_timeout 60s、各 server に keepalive_timeout 30s、reset_timedout_connection on。**上流の応答待ちの既定を 60s に**。300s は /admin/downloads.php・8281・6001・3001・3002 だけ、地図の取り込みと配信の画面は 180s(PHP の set_time_limit 120 より長く) |
| nginx の接続数 | `limit_conn`(IP ごと): 443・3001 は 100(学校の NAT を考えて広め)、6001・8281・3002・8025 は 50。超えたら 429 |
| nginx の回数 | 未認証で重い口にメソッドを問わない `km_api`(240r/m、burst 120): app-stats・app-avatar・floor-image・download・map-data・logto_me。`/api/app-map.php` は `km_appmap`(30r/m、burst 30) |
| Apache | Timeout 300 → 60。PATH_INFO を断つ |
| PHP | `docker/php/99-limits.ini`: max_execution_time 30、default_socket_timeout 15(memory_limit は変えない)。公開の JSON API は本文 64KB を超えたら 413(アプリの設定は 128KB、webhook は 1MB。Content-Length の無い本文も上限+1 バイトまでしか読まない)。アカウント画像は縦横 2048px まで |
| MariaDB | `--max-connections=120`。PDO の接続待ち 5 秒、Web のときだけ `SET SESSION max_statement_time = 15`(**グローバルには設定しない** —— mysqldump の控えが止まるため) |
| Soketi | 接続 300、バックエンドのイベント 50/秒、クライアントのイベント 5/秒、読み取り 20/秒、クライアントイベントは無効(src と Android で送っていないことを grep で確認)。USER node |
| コンテナの上限 | mem_limit: logto 512m / mariadb 512m / postgres 256m / web 384m / reverse-proxy 128m / soketi 256m / phpmyadmin 256m / mailpit 128m / mailserver 192m / certbot 128m。pids_limit は各 200(logto 256) |
| JS | 取得に 15 秒の打ち切り(AbortController)。死活監視の自動更新は、前回が終わってから次を予約し、失敗が続くと倍々に延ばす(上限 5 分、±20% のゆらぎ)、タブが隠れている間は止める |
| Android | 全 HTTP を接続 10 秒・読み取り 20 秒。応答の上限(地図 20MB・JSON 1MB・画像 2MB)。画像は縮小デコード。人数とランキングの送信は失敗時に指数的に待ち(ゆらぎあり)、連打は間引く |

**手元で確かめたこと:** nginx のテンプレートを展開して、括弧の釣り合い・一度しか書けない指示の重複 0 件・http に keepalive_timeout が無い(公式イメージの nginx.conf と重複して起動しないため)ことを確かめ、本番と同じイメージで `nginx -t` が通った。
services.js は DOM と fetch を差し替えた node の試験で、本文が止まっても 15 秒で戻ること、隠す→表示で余計な確認を出さないことを確かめた。

**後から入ったもの(§5.6):** web の mem_limit 384m に対して Apache の MaxRequestWorkers が 150 のままだと、重い PHP が同時に3本走るとコンテナの中で OOM が起きうる。
apache-km.conf で **MaxRequestWorkers 30** に絞った。admin/monitor.php の自動更新は services.js の startAuto / stopAuto に、admin/chat.php と kanban.php の fetch には打ち切りが入った。
**30 で足りるか、上限に張り付かないかは、配備後に `docker stats` で見る(§6 の 20)。**

### 5.5 地図の配信を「Website の正本」と「イベント用の正本」に分ける

| 何を | どうしたか |
|---|---|
| 配信 ID | **`kosen-main`**(Website の正本。DB から作る、今までの動き)と **`kosen-event`**(イベント用の正本。管理画面で JSON を添付)。どちらも版・期限・停止・チェックサム・アクセスコードを持つ。設定の形は今までの `maps[slug]` と `codes[].slug` のまま。古い設定もそのまま読める |
| 管理画面「アプリへ地図を配信する」 | 2つの区画。イベント用は JSON を添付(10MB まで)して中身を確かめ(地点 0 件・型違いは断る)、版を上げる。「Website のイベント(通行止め)も折り込む」は既定オフ。監査ログに残す。上限を超えた添付は「セッションの有効期限」ではなく「ファイルが大きすぎます」と出す |
| 配信先の無いコード | **本番の TEST1 のようなコード**を警告つきで一覧に出し、kosen-main・kosen-event への付け替えと「止める」を付けた |
| 照合を安く | コードを作るときに HMAC の lookup を保存し、まず lookup で引く。lookup の無い古いコードだけ bcrypt で照合し(1回の要求で最大 10 件)、当たったら lookup を書き足す(変わるときだけ書く) |
| 失敗ロック | **先に数えてから照合する**(1文の INSERT … ON DUPLICATE KEY UPDATE)。アプリは 30 回/15 分(学校の共有回線)、Web 地図の合言葉は 8 回のまま。**正しいコードはロック中でも通す**。アプリは成功しても失敗回数を消さない。IPv6 は /64 で数える |
| 版の比較 | 端末は `haveMapId` を送る。サーバーは配信 ID が同じときだけ版で「最新です」を返す。haveMapId の無い古いアプリは kosen-main のときだけ従来どおり |
| CSP | map-publish.php の onsubmit= と nonce 無しの <script> をやめた(本番の CSP で動いていなかった) |
| 配布ファイルの画面 | 配信カードに正本の種類を出し、停止・削除が2つの配信 ID を扱える |
| Android | mapId が違えば版に関係なく置き換え対象(端末の編集の上書き確認を通す)。QR を読んだら入力欄に入れるだけ |

**手元で確かめたこと:** 配信 ID・lookup と bcrypt・付け替え・版の比較・添付の検証・30 回のロック・IPv6 /64・DB が無いときの予備ファイルを、ローカルの試験(ALL PASS)と見直しの追加試験(8 件)で確かめ、同じ内容を check.php の `hardening` に入れた(本番の形 = TEST1 の bcrypt だけの設定で、一覧に出る・bcrypt で当たる・付け替えで消えることを含む)。
Android の MapPackageTest に haveMapId と mapId の切り替えを足した。
**失敗ロックの SQL は MariaDB で実行していない**(文面・プレースホルダの数・代入の順を読んで確かめた)。管理画面はブラウザで描画していない。

### 5.6 担当の間の引き継ぎ(後から済んだもの)

実装の報告(integration.handoff)で「担当の持ち物の外」として残っていたもの。**この報告を作っている間に済んだことを、ファイルを検索して確かめた**(2026-09-14 14:34〜14:43 の更新)。
中身の読み込みと試験はこの報告ではしていない。check.php は通っている(951 件)。

| 持ち主 | 何を | ファイルで確かめたこと |
|---|---|---|
| php | admin/monitor.php の自動更新を `KmServices.startAuto` / `stopAuto` に | startAuto の呼び出しがある |
| php | admin/chat.php の fetch 3 か所と admin/kanban.php の fetch に打ち切り | どちらにも AbortController がある |
| php | lib/security-notice.php の種類に fail2ban と logto | fail2ban の種類がある |
| php | lib/distributables.php の利用者向けの断りを KmUserError に | KmUserError を使っている |
| map | ja.js に page.mapEditor.timeout など。en.js に監査ログの新しい操作の英訳 | ja.js に mapEditor.timeout、en.js に admin_write_denied・account_suspended_blocked・webhook_signature_failed・map_code_reassign |
| scripts | scripts/app-map-config-set.php が作るコードに lookup | lookup を書いている |
| 取扱説明書 | [09-new-host](09-new-host.ipynb) に「ドメインを変える」 | §6 に host-cert.sh issue → host-domain.sh check / apply → logto-domain.php の実行できるセルがある |
| infra | MaxRequestWorkers と mem_limit の釣り合い | docker/php/apache-km.conf に MaxRequestWorkers 30 |

入口の [11-getting-started](11-getting-started.ipynb) も同じ時間に足された。

## 6. 本番で実行すること(順番どおり)

**番号 1〜25 は実装の報告(impl-result.json の integration.handoff)の番号と同じ。** 分からない出力が出たら、先へ進まずに止める。

| 段 | 番号 | 本番 |
|---|---|---|
| 6.1 配備の前 | 0〜3 | 変えない |
| 6.2 配備 | 4〜9 | **変える** |
| 6.3 配備の後の確認 | 10〜20 | 変えない |
| 6.4 配備の後の作業 | 21〜25 | **変える** |

**Android(25)はサーバーの配備が済んでから。** 新しいアプリは haveMapId を送り、サーバーの新しい照合を前提にしている。

### 6.1 配備の前(読むだけ)

#### 0) 手元の検査と下見

`すべて通過 (951 件)` と出ること。下見は接続もしない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
php ..\src\scripts\check.php
.\deploy-to-host.ps1 -WhatIfOnly

#### 1) .env の KM_APP_URL の形

**6001 の Origin の検査は、ブラウザが送る Origin と KM_APP_URL を文字列で完全一致させて比べる。**
`https://ito8795.com` の形(末尾の `/` も `:443` も無い)であること。違うと管理画面のチャットと死活監視が「未接続」になる。
(この2行は秘密ではない。**.env のほかの行は出さない。**)

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
grep -E '^(KM_DOMAIN|KM_APP_URL)=' .env || echo "KM_DOMAIN / KM_APP_URL の行がありません"
grep -cE '^KM_API_RESOURCE=' .env | sed 's/^/KM_API_RESOURCE の行の数: /'

#### 2) cron がまだ版 5 であること

1行目の版を見る(読むだけ)。スクリプト本体の確認は sudo が要るので別の窓。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
head -1 /etc/cron.d/kosenmap-updates
stat -c '%U:%G %a %n' /etc/cron.d/kosenmap-updates

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-updates-setup.sh --path /opt/kosenmap"

#### 3) 証明書の状態と、更新方式の揃え方の下見

`fix-conf --dry-run` は試験用の発行元で1回通すだけで、設定は変えない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
sh scripts/host-cert.sh status </dev/null
echo
sh scripts/host-cert.sh fix-conf --dry-run </dev/null

### 6.2 配備(本番が変わる)

#### 4) 控えを取る

**配備で壊れたときに戻す先。** 23) の Logto の書き換えの前提でもある。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal --confirm "本番で控えを取ります(sudo のパスワードを聞かれます)"
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-backup.sh --path /opt/kosenmap"

#### 5) 配備して up -d

[02-deploy](02-deploy.ipynb) と同じ。済むと後片付け(`host-setup.ps1 -Fix`)が自動で走る。
compose.yaml を変えているので `-Action up`(**変わったコンテナが作り直される。数十秒サイトが途切れる**)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番へ配備して docker compose up -d します(変わったコンテナが作り直されます)"
.\deploy-to-host.ps1 -Action up -Yes

#### 6) ホストで compose を確かめ、Soketi を作り直し、nginx を確かめ、Mailpit を止める

**6a. 設定が読めるか**(読むだけ)

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose config -q && echo "compose の設定は読めます"

**6b. Soketi のイメージを作り直す**(Dockerfile に `USER node` を足したため。5) の up では作り直されない)

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番で Soketi のイメージを作り直します(まだ入れ替えません)" --timeout 900
docker compose build soketi

**6c. 立ち上げ直す**(作り直した Soketi に入れ替わる。管理画面のチャットと監視が一瞬切れる)

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番で docker compose up -d します(変わったコンテナが作り直されます)" --timeout 900
docker compose up -d
echo
docker ps --format '{{.Names}}\t{{.Status}}' | sort

**6d. nginx の設定の検査**(読むだけ。`syntax is ok` と `test is successful`)

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose exec -T reverse-proxy nginx -t

**6e. Mailpit を止めて消す**

新しい compose.vps.yaml では Mailpit が `profiles` の内側にあるので、up しても起動しないが、**前から動いているものは残る**。
名指しできるよう `--profile mailpit` を付ける(実装の報告のコマンドに、この指定を足した)。メールは 0 通(実測)なので失うものは無い。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番の Mailpit を止めて消します(本番のメールは mailserver が送るので影響しません)"
docker compose --profile mailpit stop mailpit
docker compose --profile mailpit rm -f mailpit
docker ps -a --filter name=km-mailpit --format '{{.Names}} {{.Status}}'
echo "(上に何も出なければ消えています)"

#### 7) 所有権と実行ビット

5) の後に自動で走っているはずだが、host-cert.sh・host-domain.sh に実行ビットが付いたことを確かめる意味でもう一度。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番の所有権と権限を直します"
.\host-setup.ps1 -Fix

#### 8) cron を版 6 にする

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal --confirm "本番の cron と自動更新の設定を版 6 に書き直します(sudo のパスワードを聞かれます)"
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-updates-setup.sh --path /opt/kosenmap --fix"

版が 6 になり、`host-cert.sh` の行があること。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
head -1 /etc/cron.d/kosenmap-updates
grep -n 'host-cert.sh' /etc/cron.d/kosenmap-updates

#### 9) 証明書の更新方式を webroot に揃える

試験用の発行元で1回通してから書き換える(3) の下見と同じ流れ)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番の証明書の更新設定(renewal/*.conf)を webroot に書き換えます" --timeout 600
sh scripts/host-cert.sh fix-conf </dev/null
echo
sh scripts/host-cert.sh status </dev/null

### 6.3 配備の後の確認(読むだけ)

#### 10) PATH_INFO 付きの URL が 404 になる

外(この PC)から叩く。上の3つが 404、最後の `/contact.php` は 200。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
foreach ($path in '/contact.php/x', '/logto-client.php/x', '/admin/index.php/a.css', '/contact.php') {
    $status = curl.exe -s -o NUL -w '%{http_code}' "https://ito4.jp$path"
    '{0,-26} {1}' -f $path, $status
}

#### 11) PHP・Apache・MariaDB の設定が効いている

`default_socket_timeout => 15`、`session.use_strict_mode => On`、`max_execution_time => 30`、Apache に `AcceptPathInfo Off` と `Timeout 60`、MariaDB は `0 120`(local_infile と max_connections)。
MariaDB のパスワードはコンテナの中の環境変数から MYSQL_PWD で渡す(コマンド行にも画面にも出ない)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose exec -T web sh -c 'php -i | grep -E "^(default_socket_timeout|session\.use_strict_mode) "' </dev/null
# max_execution_time は php -i(コマンドライン)だと常に 0 と出る。Web で効く値は設定ファイルで見る
docker compose exec -T web sh -c 'grep -hE "^max_execution_time" /usr/local/etc/php/conf.d/99-limits.ini' </dev/null
docker compose exec -T web sh -c 'grep -hE "^[[:space:]]*(AcceptPathInfo|Timeout)" /etc/apache2/conf-enabled/km.conf' </dev/null
docker compose exec -T mariadb sh -c 'MYSQL_PWD="$MARIADB_ROOT_PASSWORD" exec mariadb -uroot -N -e "SELECT @@local_infile, @@max_connections"' </dev/null

#### 12) コンテナの上限

mem が 0 でないこと(mailserver 以外は `opt=[no-new-privileges:true]`)。soketi は `user=node`。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
for c in $(docker ps --format '{{.Names}}' | sort); do
  docker inspect "$c" --format '{{.Name}}  mem={{.HostConfig.Memory}}  pids={{.HostConfig.PidsLimit}}  opt={{.HostConfig.SecurityOpt}}  user={{.Config.User}}'
done

#### 13) 管理系の3ポートが IPv4 だけで待つ

`0.0.0.0:8281` などが出て、`[::]:8281` などが**出ない**こと。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
ss -ltn | grep -E ':(8281|3002|8025)[[:space:]]' || echo "該当する待ち受けがありません"

#### 14) 6001 が知らない Origin を断る

**画面で:** 管理画面のチャットと死活監視が「接続済み」になること(ここが「未接続」なら 1) の KM_APP_URL の形を疑う)。

下のセルは、知らない Origin で WebSocket の開始を頼み、**403** が返ることを見る。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
$status = curl.exe -s -o NUL --max-time 5 -w '%{http_code}' `
    -H 'Origin: https://evil.example' -H 'Connection: Upgrade' -H 'Upgrade: websocket' `
    -H 'Sec-WebSocket-Version: 13' -H 'Sec-WebSocket-Key: dGhlIHNhbXBsZSBub25jZQ==' `
    'https://ito4.jp:6001/app/check'
"知らない Origin への応答: $status(期待は 403)"

#### 15)〜19) 画面で確かめる

| 番号 | どこで | 何が見えれば良いか |
|---|---|---|
| 15 | Logto Console → Webhooks → テスト送信 | 2xx が返る。管理画面のタイムラインに `webhook.signature_failed` が出ない |
| 16 | https://ito8795.com:8281/ | **root では入れない**(Main では入れる) |
| 18 | 管理画面「地図の取り込み」を、地点の多い地図で1回 | 504 にならず完了の表示まで戻る |
| 19 | 公開の問い合わせフォームから1通 | 送れる(落ちたら web のログに「ホスト名が違います」「action が違います」) |
| — | 管理画面「アプリへ地図を配信する」でコードを止める操作(確かめたら元に戻す) | 確認ダイアログが**1回だけ、日本語の文で**出る |

15) と 19) のあと、web のログに webhook と reCAPTCHA の失敗が出ていないかを見る。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose logs --since 30m web 2>&1 | grep -iE 'webhook|recaptcha|ホスト名が違います|action が違います' | tail -20
echo "(何も出なければ失敗の記録はありません)"

#### 17) 3001 の HSTS が1本だけ

3001 と 443 それぞれで `strict-transport-security` の行が **1 本**。3001 に includeSubDomains が付いていないこと。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
foreach ($url in 'https://ito4.jp:3001/oidc/.well-known/openid-configuration', 'https://ito4.jp/') {
    $headers = curl.exe -s -D - -o NUL $url
    $sts = @($headers | Select-String -Pattern '^strict-transport-security')
    '{0}  HSTS の行: {1} 本' -f $url, $sts.Count
    $sts | ForEach-Object { '    ' + $_.Line.Trim() }
}

#### 20) 上限に張り付いていないか(数日は毎日)

MemPerc が 90% を超え続けるか、OOMKilled=true が出たら、web の MaxRequestWorkers(いま 30)をさらに絞るか mem_limit を上げる。逆に混む時間に 503 や待ちが増えるなら 30 が少なすぎる(§5.4)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker stats --no-stream --format 'table {{.Name}}\t{{.MemUsage}}\t{{.MemPerc}}\t{{.PIDs}}'
echo
for c in $(docker ps --format '{{.Names}}' | sort); do
  printf '%s OOMKilled=%s\n' "$c" "$(docker inspect -f '{{.State.OOMKilled}}' "$c")"
done

### 6.4 配備の後の作業(本番が変わる)

#### 21) TEST1 を付け替える

**画面で:** 管理画面「アプリへ地図を配信する」の上の「配信先の無いアクセスコード」で、TEST1 を **kosen-main** に付け替える。
配ったコードを使わないなら「止める」を押し、kosen-main で新しく作る。**保存できればそれで終わり。**

まず今の設定を見る(配信 ID と、コードの配信先・lookup の有無だけ。ハッシュは出さない)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose exec -T web php -r '
$c = require "config/app-map.local.php";
foreach (($c["maps"] ?? []) as $slug => $m) { echo "配信 ", $slug, PHP_EOL; }
foreach (($c["codes"] ?? []) as $x) { echo "コード → ", ($x["slug"] ?? "?"), "  ", (isset($x["lookup"]) ? "lookup あり" : "lookup なし"), PHP_EOL; }
' </dev/null

**「配信設定を書き込めませんでした」と出たときだけ**、持ち主を直す。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番の config/app-map.local.php の持ち主を www-data にします"
# 持ち主だけを www-data にする。**グループ(km)は変えない** —— host-setup.sh と同じ www-data:km 640 にそろえる
docker compose exec -T -u root web chown www-data config/app-map.local.php </dev/null
docker compose exec -T -u root web chmod 640 config/app-map.local.php </dev/null
docker compose exec -T web ls -l config/app-map.local.php </dev/null

#### 22) fail2ban を動かす

まず、なぜ止まったかと、設定の検査の結果を見る(何も変えない)。`fail2ban-client -t` が誤りを出したら、**その行を直してから**次へ。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo systemctl status fail2ban --no-pager -l | head -20; sudo journalctl -u fail2ban -n 40 --no-pager; sudo fail2ban-client -t"

検査が通ったら、起動して自動起動にする。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal --confirm "本番で fail2ban を起動し、自動起動にします(sudo のパスワードを聞かれます)"
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo systemctl enable --now fail2ban && sudo fail2ban-client status"

#### 23) Logto Console(admin テナント)の MFA を必須にする

**§7 の「Logto Console の MFA の進め方」を決めてから。** やるなら次の順:

1. 4) の控えがあることを確かめる
2. **認証アプリ(TOTP)を手元に用意する**。Console に入ったままのブラウザを1つ残しておく(締め出されたときの戻り道)
3. 下の読むだけのセルで今の値を控える
4. 書き換えのセル → Console に入り直し、登録する

今の値(tenant ごとの MFA の方針と要素。パスワード方針とロックも出す。**値に秘密は含まれない**):

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose exec -T postgres sh -c 'exec psql -U "$POSTGRES_USER" -d "$POSTGRES_DB" -At -F " | "' <<'SQL'
select tenant_id, mfa->>'policy', mfa->'factors' from sign_in_experiences order by tenant_id;
select tenant_id, password_policy::text from sign_in_experiences order by tenant_id;
select tenant_id, sentinel_policy::text from sign_in_experiences order by tenant_id;
SQL

admin テナントの MFA を Mandatory(Totp・WebAuthn)にする。
実装の報告のコマンドは `mfa` を丸ごと置き換える形だったが、**ほかの鍵を消さないよう、今の値に重ねる形**(`||`)にした。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の admin テナント(Console)の MFA を必須にします。認証アプリの用意と控え(4)は済みましたか"
docker compose exec -T postgres sh -c 'exec psql -U "$POSTGRES_USER" -d "$POSTGRES_DB" -v ON_ERROR_STOP=1' <<'SQL'
update sign_in_experiences
   set mfa = coalesce(mfa, '{}'::jsonb) || jsonb_build_object('policy', 'Mandatory', 'factors', jsonb_build_array('Totp', 'WebAuthn'))
 where tenant_id = 'admin';
select tenant_id, mfa->>'policy', mfa->'factors' from sign_in_experiences where tenant_id = 'admin';
SQL

Console に入り直しても登録の画面が出ないとき(Logto が設定を覚えたままのとき)だけ、Logto を立ち上げ直す。**数秒サインインが止まる。**

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番の Logto を再起動します(数秒サインインが止まります)"
docker compose restart logto

#### 24) セキュリティの見張りで確かめる

fail2ban と Logto Console の MFA の ★ が消えていること。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-security-check.sh --path /opt/kosenmap"

#### 25) Android をリリースビルドして実機で確かめ、それから配る

**サーバーの配備が済んでから。** 両フレーバーを作る(手元の Test に APK ができる)。

実機で1回ずつ確かめること:
1. ログイン済みの旧版から更新しても**ログアウトされない**(平文のトークンを Keystore へ移す)
2. ログアウトしても**落ちない**
3. QR を読んでも**自動で送らない**
4. ランキングの**同意ダイアログが出る**
5. 来場者ビルドでスタッフがログアウト → ログインし直して地図を取ると、**氏名とスタッフ限定の地点が戻る**

🟡 **手元が変わる** —— この PC にファイルを作ります。本番には触れません。

In [ ]:
%%ps --timeout 1800
$env:JAVA_HOME = 'C:\Program Files\Eclipse Adoptium\jdk-21.0.12.101-hotspot'
Set-Location 'C:\Users\itota\Documents\Test'
.\gradlew.bat assembleVisitorRelease assembleAdminRelease --console=plain "-PkosenmapKeystoreProperties=C:\Users\itota\Documents\keystore.properties"
Get-ChildItem .\app\build\outputs\apk -Recurse -Filter *.apk | Select-Object FullName, Length, LastWriteTime

## 7. 利用者の判断待ち

**今回は手を付けていない。** 直し方に選択肢があるか、戻せない変更を含むもの。上から大事な順。

| # | 何を決めるか | 選択肢 | 決めないと |
|---|---|---|---|
| 1 | **Logto Console の MFA の進め方** | (a) §6 の 23 で SQL を当てて Mandatory にする(Console の画面は default テナントの設定を扱う。admin テナントの MFA を画面で変えられるかは確かめていない)/(b) 今のまま(IP 制限 + ゲートの MFA に任せる)。どちらでも、パスワード方針と総当たりロック(sentinel)を足すかも決める | Console の資格情報が漏れたら、最後の壁が無い |
| 2 | **MariaDB の `root@%` と `Main@%`** | root を `localhost` だけにする(phpMyAdmin の root ログインは既に塞いだ)/ Main を DML(SELECT・INSERT・UPDATE・DELETE)と必要な DDL だけにする。**アプリは管理画面の汎用テーブル編集で CREATE・ALTER・DROP を使う**ので、絞る前にそれを確かめる | SQLi が1つ見つかったときの被害が Kosen_map の全権になる |
| 3 | **Logto の Postgres 利用者を SUPERUSER でないロールへ** | Logto 用のロールを作り、データベースの持ち主を移して DB_URL を替える。**Logto の起動時の移行(alteration)が権限を要するので、使い捨ての環境で試してから** | Logto に SQLi が出たとき COPY … PROGRAM でコンテナ内の実行まで届く |
| 4 | **Soketi の Node 16** | node:18-bookworm で動くか使い捨ての環境で試す(Node 20 では起動しない)/ Soketi を別の実装に置き換える / 今のまま(node 利用者と上限で囲った) | パッチの出ないランタイムを公開し続ける |
| 5 | **Test.zip**(DB 接続設定と local.properties を含む) | Test/Old/ へ退避する / 秘密を抜いて作り直す。**中の DB 資格情報をローテートの対象に入れる** | 共有やアップロードで一緒に出ていく |
| 6 | **ランキングに溜まった行**(既定 ON だった 2026-08-29〜09-14 に、同意なく入った表示名) | 利用者ランキングの行を消す / 残す(端末は次のログインで改めて尋ね、断られたら消す。ログインしない端末の行は残る)。`src/scripts/reset-app-ranking.php` は引数なしなら一覧だけ | 同意の無い表示名が公開のまま |
| 7 | **app/build (1).gradle.kts と未使用の4ファイル**(MapLayerRenderer.kt・NodeInfoDialog.kt・file_paths.xml・MapLayerRendererTest.kt) | Test/Old/ へ退避する / 残す | 取り違えで LAN 向け・難読化なしのリリースに戻る恐れ |
| 8 | allow-admin.conf の `allow 172.16.0.0/12` | SSH トンネル用の唯一の逃げ道なので残した。外すなら別の戻り道(固定 IP の許可)を先に作る | devnet の中の侵害されたコンテナから 172.x として通る |
| 9 | ネットワークの分割・read_only・cap_drop・src の :ro・全イメージの digest 固定・管理画面を別オリジンに・セッション Cookie を __Host- 付きに | 範囲外として残した。__Host- は全員が一度ログアウトされる | 多層防御の底上げが残る |
| 10 | D:\Backups の ACL(Authenticated Users が変更できる) | 本人と SYSTEM だけにする | 同じ PC の別の利用者が開いた控えを読める |
| 11 | km が docker グループ経由で実質 root / 控えの暗号に認証が無い | 範囲外。手元 PC の鍵にパスフレーズを付けるかも含めて | 手元 PC の侵害がホストの root に届く |
| 12 | Logto の KosenAPP(Native)の refresh_token の回転と絶対期限 | Console で有効にする(アプリ側は対応済み) | 抜かれたトークンが長く使える |
| 13 | reCAPTCHA の管理画面の「ドメイン名の検証」 | 有効か確かめる(サーバー側でもホスト名を照合するようにした) | — |
| 14 | web のイメージの pdo_pgsql | 外すと PHP の版まで上がりうる(php:8.4-apache は動くタグ)ので残した | 使わない拡張が残る |

**2026-09-15 に 14 項目すべての進め方が決まった。** 決まったこと・使い捨ての環境で確かめたこと・本番への当て方は [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb)。

判断の材料を読むだけで集めるセル(**値を変えない。パスワードはコンテナの環境変数から渡し、画面に出さない**):

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
echo "== 2) MariaDB の利用者と Main の権限 =="
# SHOW GRANTS はパスワードのハッシュも出す。**ノートに残さないよう伏せてから表示する**
docker compose exec -T mariadb sh -c 'MYSQL_PWD="$MARIADB_ROOT_PASSWORD" exec mariadb -uroot' <<'SQL' | sed "s/IDENTIFIED BY PASSWORD '[^']*'/IDENTIFIED BY PASSWORD '<伏せ>'/"
SELECT user, host FROM mysql.user ORDER BY user, host;
SHOW GRANTS FOR 'Main'@'%';
SQL
echo
echo "== 3) Logto の Postgres 利用者 =="
docker compose exec -T postgres sh -c 'exec psql -U "$POSTGRES_USER" -d "$POSTGRES_DB" -At' <<'SQL'
select rolname, rolsuper, rolcreaterole, rolcreatedb from pg_roles where rolname = current_user;
SQL
echo
echo "== 4) Soketi =="
docker compose exec -T soketi sh -c 'node --version; id -un' </dev/null
echo
echo "== 6) ランキングの利用者の行の数 =="
docker compose exec -T mariadb sh -c 'MYSQL_PWD="$MARIADB_ROOT_PASSWORD" exec mariadb -uroot -N Kosen_map' <<'SQL'
SELECT COUNT(*) FROM km_map_ranking_users;
SQL

手元の5) と7) の対象(大きさと日付だけ。中身は開かない):

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
$test = 'C:\Users\itota\Documents\Test'
@('Test.zip', 'app\build (1).gradle.kts',
  'app\src\main\java\com\ito\kosenmap\MapLayerRenderer.kt',
  'app\src\main\java\com\ito\kosenmap\NodeInfoDialog.kt',
  'app\src\main\res\xml\file_paths.xml') | ForEach-Object {
    $p = Join-Path $test $_
    if (Test-Path -LiteralPath $p) { $i = Get-Item -LiteralPath $p; '{0,-60} {1,12:N0} バイト  {2:yyyy-MM-dd}' -f $_, $i.Length, $i.LastWriteTime }
    else { '{0,-60} (ありません)' -f $_ }
}

## 8. この診断の限界

**「問題なし」「直した」は、次の条件の下での話。**

| 見ていないもの | どう困るか | 埋めるには |
|---|---|---|
| **外部の CVE データベース**(NVD・GitHub Advisory・OSV)との照合 | composer.lock・Gradle の依存・コンテナイメージに既知の脆弱性があっても分からない。カタログの 2026 年の CVE 番号と修正版も確かめていない | `composer audit`、Trivy / Grype でイメージ、Gradle の依存の照合。Logto の版を上げるときは公式のリリースノートとアドバイザリを読む |
| **ブラウザでの動き** | 管理画面の新しいフォーム(ログアウト・確認ダイアログ・地図の2区画・付け替え)、write スコープの 403、CSP の違反が画面で出ないか | §6 の 14〜19 |
| **Android の実機** | ログアウト時のクラッシュの修正、Keystore への移行、失効の送り直し、同意ダイアログ、R8 を掛けたリリース APK | §6 の 25 |
| **本番での動き** | 手元の検査は文字列と関数の呼び出し。本番と同じイメージで `nginx -t`・`docker compose config` は通したが、**負荷の値(接続数・回数・mem_limit)が本番の利用で足りるかは測っていない** | §6 の 20 を数日 |
| **MariaDB での SQL の実行** | 失敗ロックの INSERT … ON DUPLICATE KEY UPDATE は文面を読んだだけ | 配備後にアクセスコードを数回わざと間違えて、ロックと解除を見る |
| **Logto の中** | Console の格納型 XSS、利用者の列挙、M2M ロールの範囲、シード管理者、停止した利用者のトークンが Account API に通るか | Logto の監査ログと Console の設定を人が見る |
| **侵入試験(DAST)** | 実際に攻撃を投げて確かめていない。ZAP などは走らせていない | 使い捨ての環境で |
| **読んでいないコード** | server: map-editor.php・kanban.php・monitor.php の本文、map-edit.php の後半、app-map-convert.php の全体、scripts の ps1/sh の引数の組み立て。Android: 編集ダイアログ・描画・測位の多くのファイル、androidTest | 観点ごとの「見た範囲」(audit-coverage)に書いた |
| **DNS・外部のサービス** | 使っていないサブドメインのレコード、reCAPTCHA の管理画面、メールの DKIM・SPF・DMARC | 人が見る |
| **host-domain.sh の丸ごとの試験** | この PC の sh が fork に失敗し、部分(awk と grep)だけ試した | ドメインを変える前に、使い捨てのホストで check と apply を通す |
| **反証・可能性の判断** | 検証の担当の読みに依る。「可能性」は §3.3 の決め手を確かめるまで確定しない | §3.3 |

診断は2026-09-14 の作業ツリーと、同じ日に読むだけで測った本番の値に基づく。**本番は SSH・curl・配備のどれでも変えていない。**